<!-- track-identity-card -->
# Identity transfer across cars

| | |
|---|---|
| Pipeline step | `07_cross_car.ipynb` |
| Manuscript section | 4.4 |
| Copied from | `notebooks/NB16_cross_car_identifiability_v5_3_2026-07-22.ipynb` |
| Source sha256 | `53eacc7f8c0906f77c986ca44bd49914` |

**Reads**

- `data/features/*`
- `data/processed/tier3_large_3t3c/*`

**Writes**

- `results/cross_car_<policy>/nb16_info.json`
- `results/cross_car_<policy>/repeatability_per_metric.csv`
- `results/cross_car_<policy>/identifiability_pooled.csv`

Produces the repeatability coefficients and the cross-car identification results of Section 4.4. Runs with seed=42, recorded in the output. NOTE: an earlier revision (v5.2) unpacked the FDR helper incorrectly and inverted the rejection logic; this revision fixes it and adds the fdr_rej field.

> Copied from the working notebook named above. Two changes were made to it: this identity card and the bootstrap cell that follows it were added, and the hard-coded data paths were replaced with the root that the bootstrap cell resolves. The analysis code is unchanged.


# NB16 — Cross-Car Driver Identifiability & Repeatability (v4)

**Amac:** ACGym populasyonunda "surus parmak izi arac degisince suruyu tanimliyor mu?"
sorusunu iki bagimsiz cerceveyle test etmek.

**Neden bu notebook var:** K2 katkisi bugune kadar NB14 hucre 21'e dayaniyordu — **n=1 surucu**
(Efe), 4 ACC GT3 araci, 11 tur, tek pist. Leave-one-car-out testinde cokuyor
(audi_r8 cikarilinca B2 CV %107.2 -> %37.0, B3 %104.0 -> %46.5). Ayrica CV, boyut skorunun
duz ortalamasi uzerinden hesaplandigi icin fiilen tek metrigi olcuyor
(B2 skorunun %87.5'i `mean_slb_dist`). Bu notebook K2'yi n=18 uzerinde yeniden kurar.

**Iki cerceve:**

| Katman | Yontem | Kaynak literatur | n=18'de guc |
|---|---|---|---|
| Birincil | Identification accuracy + differential identifiability, permutasyon testi | Finn vd. 2015 (Nat Neurosci); Amico & Goni 2018 (Sci Rep) | **guclu** — 4/18 dogru eslesme p<0.05 |
| Mekanizma | Metrik basina adjusted repeatability (LMM), bootstrap GA, BH-FDR | Nakagawa & Schielzeth 2010 (Biol Rev); Wolak vd. 2012 (MEE) | orta |
| Kopru | Betimsel CV | NB14 ile karsilastirilabilirlik | betimsel |

**ATIF UYARISI:** Yukaridaki kunyeler 7-adim dogrulamadan GECMEDI. Makaleye girmeden once
dogrulanacak. Bu notebook onlari yalnizca yontem gerekcesi olarak anar.

---

## ON-KAYIT (pre-registration)

Asagidaki taahhutler **veriye bakilmadan** yazilmistir. Sonuclar ne cikarsa ciksin yorum
bu tabloya gore yapilacak.

### Senaryo matrisi

| | Repeatability: bazi metrikler yuksek | Repeatability: hepsi dusuk/belirsiz |
|---|---|---|
| **Identifiability anlamli** | **S1** K2 dogrulandi + mekanizma belirlendi | **S2** Kombinasyon tanimliyor, tek metrik degil; cok-degiskenli imza bulgusu |
| **Identifiability anlamsiz** | **S3** Metrikler surucuye ozgu ama 4-boyut toplulastirmasi sinyali yok ediyor -> bulgu fingerprint mimarisi hakkinda | **S4** K2 duser; negatif bulgu olarak Discussion'da kalir |

### Karar esikleri (onceden sabit)

- Identifiability anlamli sayilir: permutasyon p < 0.05 (10.000 tur)
- Repeatability "yuksek" sayilir: R_adj > 0.50 **ve** BH-FDR duzeltilmis p < 0.05
- Duyarlilik basarisiz sayilir: leave-one-car-out'ta ana sonucun isareti degisiyorsa
- Coklu test: 19 metrik icin Benjamini-Hochberg FDR (Bonferroni degil — metrikler korele,
  brake_prs<->trail_prs rho=0.926)

### Ek taahhutler

1. Efe'nin n=1 verisi K2'nin kaniti olarak KULLANILMAYACAK. En fazla Discussion'da
   "sinif-ici (dort GT3) vs sinif-otesi (GT3/formula/Miata) varyasyon" karsilastirmasi olur.
2. Hangi senaryo cikarsa ciksin raporlanacak. Duyarlilik testi cokerse gizlenmeyecek.
3. Sonuclar `nb16_info.json` dosyasina yazilacak; paper_numbers oradan beslenecek.

In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 1 — Kurulum + a priori hassasiyet (veriye BAKMADAN)
# ══════════════════════════════════════════════════════════════
import numpy as np, pandas as pd, json, warnings, time
from pathlib import Path
from itertools import combinations
from scipy.signal import savgol_filter
from scipy.stats import norm
import matplotlib as mpl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

RNG_SEED = 42
rng_global = np.random.default_rng(RNG_SEED)

# ╔══════════════════════════════════════════════════╗
# ║  KOSU AYARI — SADECE BURAYI DEGISTIR             ║
# ╚══════════════════════════════════════════════════╝
POLICY     = "P_INC"   # "P_EXC" = A-blogu HARIC (ON-KAYIT BIRINCILI, n=13 kisi)
                       # "P_INC" = A-blogu DAHIL (duyarlilik, n=16 kisi)
DO_RUNG1   = False     # Basamak 1 (gun-ici tur yarilama) — segmentasyonu ~3x yavaslatir

# --- ORTAK KIMLIK POLITIKASI (NB11 v3 / NB7A7 v2 / NB15 v3 ile AYNI liste) ---
EXCLUDE_HARD = [
    "20240213_000000_ACAI",   # oyun-ici yapay surucu
    "20240308_ensemble",      # RL politika ciktisi
    "20240501_MPC",           # klasik kontrol baseline (veri kumesi belgesi)
]
ABLOCK = [
    "20240410_A_12_123",
    "20240410_A_21_231",
    "20240411_A_12_312",
]
assert POLICY in ("P_INC", "P_EXC"), "POLICY 'P_INC' ya da 'P_EXC' olmali"
EXCLUDE_IDS    = EXCLUDE_HARD + ([] if POLICY == "P_INC" else ABLOCK)
POL_SUF        = "_inc" if POLICY == "P_INC" else "_exc"
DEGENERATE_IDS = EXCLUDE_IDS          # geriye donuk ad

# --- KIMLIK BIRIMI: KISI (v4'te oturum-gunuydu) ---
import re


def person_of(driver_id):
    """Oturum-gunu kimliginden KISI etiketi.
    'YYYYMMDD_XX' ve 'YYYYMMDD_HHMMSS_XX' -> 'XX'.

    UYARI (makale metnine girecek): bu bir ANALIST CIKARIMIDIR, ground-truth
    kimlik degil. Telemetri dosyasindaki driver_name alani kimlik tasimaz
    (tek ad 15 farkli driver_id'ye atanmis: oturumu toplayanin hesabi).
    """
    return re.sub(r'^(\d+_)+', '', str(driver_id))

ID_COL = 'person'      # tum kimlik makinesi bunu kullanir

# --- S1: A PRIORI metrik sinifi (veriye BAKILMADAN sabit) ---
DIMENSIONLESS = ['speed_loss_eff', 'mean_mc_speed_ratio', 'trail_braking_ratio',
                 'pct_lift_coast', 'pct_flat_out', 'pct_heavy_braking', 'pct_trail_braking']

# --- S2: turetilmis oransal metrikler (viraj duzeyinde once oran, sonra ortalama) ---
DERIVED_METRICS = ['r_exit_over_apex', 'r_slb_share', 'r_coast_share',
                   'k_decel_use', 'r_apex_position']

PROJECT      = TRACK_ROOT
TIER_DIR     = PROJECT / "data" / "processed" / "tier3_large_3t3c"
FEATURES_DIR = PROJECT / "data" / "features"
# v5: politika ekli cikti — iki politika birbirini EZMEZ, v4 arsivi korunur
OUT_DIR      = PROJECT / "results" / ("cross_car" + POL_SUF)
FIG_DIR      = OUT_DIR / "figures"
for d in (OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

TRACKS = ['monza', 'barcelona', 'red_bull_ring']
CARS   = ['bmw_z4_gt3', 'dallara_f317', 'ks_mazda_miata']

# NB11 ile AYNI dislama listesi (tutarlilik icin)
# Duyarlilik calistirmasinda ek olarak cikarilacak supheli kimlikler
SUSPECT_PATTERNS = ["_A_12_", "_A_21_"]          # kaynak belirsiz on ekler
SUSPECT_TIMESTAMPED = True                        # YYYYMMDD_HHMMSS_XXX bicimi

DIMENSIONS = {
    'B1_Hiz':        ['mean_apex_speed', 'speed_loss_eff', 'mean_mc_speed_ratio',
                      'mean_mc_lateral', 'mean_cex_accel_rate'],
    'B2_Frenleme':   ['mean_brake_pressure', 'trail_braking_ratio', 'mean_trail_pressure',
                      'mean_slb_dist', 'mean_slb_decel', 'mean_ce_brake_turnin'],
    'B3_Strateji':   ['mean_coasting_dist', 'mean_cex_throttle_lag',
                      'pct_lift_coast', 'pct_flat_out'],
    'B4_Tutarlilik': ['apex_speed_std', 'exit_speed_std',
                      'speed_loss_eff_std', 'braking_dist_std'],
}
DIM_NAMES   = list(DIMENSIONS.keys())
ALL_METRICS = [m for v in DIMENSIONS.values() for m in v]
METRIC2DIM  = {m: d for d, ms in DIMENSIONS.items() for m in ms}

N_PERM = 10000     # identifiability permutasyonu (ucuz — matris carpimi)
N_BOOT = 200       # repeatability bootstrap (pahali — her tur bir MixedLM fit'i)
# v5 DUZELTME 1: R_PERM=200'de ulasilabilir en kucuk p = 1/201 = 0.00498,
# BH sira-1 esigi (Bonferroni) = 0.05/19 = 0.00263.  0.00498 > 0.00263 ->
# hicbir metrik FDR'yi GECEMEZDI. Mekanik imkansizlik; 500'de p_min = 0.0020.
R_PERM = 500       # repeatability permutasyon testi
N_PERM_LOO = 2000  # LOO-WCCN permutasyonu (her tur n adet whitening — pahali)
ALPHA  = 0.05

# FAST_MODE: hizli deneme icin. Nihai calistirmada False birak.
FAST_MODE = False
if FAST_MODE:
    N_PERM, N_BOOT, R_PERM = 1000, 40, 40
    print("  !! FAST_MODE ACIK — sonuclar on-izleme, makaleye GIRMEZ")

# --- Gorsel ayarlari (proje karari: Okabe-Ito + scienceplots, radar YOK) ---
OKABE_ITO = ['#000000', '#E69F00', '#56B4E9', '#009E73',
             '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
DIM_COLORS = {'B1_Hiz': '#0072B2', 'B2_Frenleme': '#D55E00',
              'B3_Strateji': '#009E73', 'B4_Tutarlilik': '#CC79A7'}
try:
    import scienceplots
    plt.style.use(['science', 'ieee', 'no-latex', 'grid'])
    _sp = True
except Exception:
    plt.style.use('seaborn-v0_8-whitegrid')
    _sp = False
mpl.rcParams.update({
    'font.family': 'serif', 'font.size': 9,
    'axes.labelsize': 9, 'axes.titlesize': 10, 'legend.fontsize': 8,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'figure.dpi': 110, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
})

def save_fig(fig, name):
    for ext in ('pdf', 'png'):
        try:
            fig.savefig(FIG_DIR / f'{name}.{ext}', bbox_inches='tight')
        except Exception as e:
            print(f"    ! {name}.{ext} kaydedilemedi: {e}")
    print(f"  kaydedildi: {name}.pdf / .png")

# --- Diverging colormap: seaborn varsa vlag, yoksa matplotlib yerlisi ---
try:
    import seaborn as sns          # noqa: F401
    CMAP_DIV = 'vlag'
except Exception:
    CMAP_DIV = 'RdBu_r'

# --- statsmodels var mi (yoksa ANOVA fallback) ---
try:
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    from statsmodels.tools.sm_exceptions import ConvergenceWarning
    warnings.simplefilter("ignore", ConvergenceWarning)
    HAS_SM = True
except Exception:
    HAS_SM = False

print("=" * 68)
print("  NB16 — CROSS-CAR IDENTIFIABILITY & REPEATABILITY")
print("=" * 68)
print(f"  Tier dizini     : {TIER_DIR}   (var: {TIER_DIR.exists()})")
print(f"  Cikti           : {OUT_DIR}")
print(f"  Metrik / boyut  : {len(ALL_METRICS)} / {len(DIM_NAMES)}")
print(f"  Permutasyon     : {N_PERM:,}   Bootstrap: {N_BOOT}   R-perm: {R_PERM}")
_est = 19 * (N_BOOT + R_PERM) * 0.055 / 60
print(f"  Tahmini repeatability suresi: ~{_est:.0f} dk "
      f"({19*(N_BOOT+R_PERM):,} model fit'i)")
print(f"  scienceplots    : {_sp}")
print(f"  diverging cmap  : {CMAP_DIV}")
if not HAS_SM:
    print()
    print("  " + "!" * 56)
    print("  ! statsmodels YOK. Repeatability TEK-YONLU ANOVA ile hesaplanacak:")
    print("  ! pist ve arac sabit etkileri MODELDE OLMAYACAK -> R asagi yonde YANLI.")
    print("  ! Cozum: conda install -c conda-forge statsmodels")
    print("  " + "!" * 56)
    print()
print(f"  statsmodels     : {HAS_SM}" + ("" if HAS_SM else "  -> ANOVA fallback kullanilacak"))

# ---------- A PRIORI HASSASIYET (Bonett 2002) ----------
def bonett_ci_width(n, k, rho, conf=0.95):
    """Beklenen ICC guven araligi genisligi. Wolak vd. 2012 (MEE 3:129-137)."""
    z = norm.ppf(1 - (1 - conf) / 2)
    if k <= 1 or n <= 1:
        return np.nan
    return np.sqrt(8 * z**2 * (1 - rho)**2 * (1 + (k - 1) * rho)**2 /
                   (k * (k - 1) * (n - 1)))

print()
print("-" * 68)
print("  A PRIORI HASSASIYET — Bonett (2002), veriye bakilmadan")
print("-" * 68)
print("  Beklenen %95 GA genisligi, n=18 kayit:")
print(f"  {'R':>6} | {'k=2':>7} {'k=3':>7} {'k=2.28':>8}")
_prec = {}
for rho in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    a, b, c = (bonett_ci_width(18, 2, rho), bonett_ci_width(18, 3, rho),
               bonett_ci_width(18, 2.28, rho))
    _prec[rho] = round(float(c), 3)
    print(f"  {rho:>6.1f} | {a:>7.3f} {b:>7.3f} {c:>8.3f}")
print("  YORUM: R~0.5'te GA genisligi ~0.64 -> orta hassasiyet.")
print("         Bu sinir ONCEDEN beyan edilmistir; sonuca gore mazeret degildir.")

# ---------- Permutasyon gucu (identifiability) ----------
print()
print("-" * 68)
print("  IDENTIFIABILITY GUCU — permutasyon null dagilimi")
print("-" * 68)
_rng = np.random.default_rng(RNG_SEED)
for n_sub in (8, 10, 12, 16, 18):
    null = np.array([np.sum(_rng.permutation(n_sub) == np.arange(n_sub))
                     for _ in range(4000)])
    thr = np.percentile(null, 95)
    print(f"  n={n_sub:>2} -> sans %{100/n_sub:>4.1f} | null ort={null.mean():.2f} "
          f"| p<0.05 icin gereken dogru eslesme: {int(np.ceil(thr))+1}")
print("  YORUM: n>=10'da 4 dogru eslesme genellikle yeterli. n<8 ise guc dusuk.")

## BLOK 2 — TDD: tahmin edicileri BILINEN CEVAPLI sentetik veriyle dogrula

Gercek veriye dokunmadan once, cevabi bilinen uc rejimde tahmin edicileri test ediyoruz.
Bu, "kod dogru mu" sorusunu gercek veriden **once** kapatir; sonuc surprizli ciktiginda
"kod mu bozuk, veri mi boyle" ikilemi olusmaz.

| Rejim | Kurulum | Beklenen identifiability | Beklenen R |
|---|---|---|---|
| NULL | surucu etkisi yok | sansa yakin, p>0.05 | ~0 |
| ORTA | surucu etkisi = artik | sanstan yuksek | ~0.5 |
| GUCLU | surucu etkisi >> artik | cok yuksek | ~0.9 |

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 2a — Tahmin ediciler (identifiability + repeatability + CV)
# ══════════════════════════════════════════════════════════════

from sklearn.covariance import LedoitWolf
from scipy.stats import levene as _levene


def loo_percentile(data, metrics, cell_cols=('car', 'track')):
    """Her metrigi kendi (arac, pist) hucresindeki LEAVE-ONE-OUT yuzdeligine cevirir.

    Neden LOO: kaydin kendisi tabani kaydirmasin. Neden yuzdelik: her monoton
    arac etkisine degismez (medyan-orandan guclu).
    """
    d = data.reset_index(drop=True).copy()
    for m in metrics:
        out = np.full(len(d), np.nan)
        for _, g in d.groupby(list(cell_cols)):
            pos = g.index.values
            v = pd.to_numeric(g[m], errors='coerce').values.astype(float)
            for j, p in enumerate(pos):
                o = np.delete(v, j)
                o = o[np.isfinite(o)]
                if len(o) and np.isfinite(v[j]):
                    out[p] = float((o < v[j]).mean())
        d[m] = out
    return d


def wccn_whiten(X, labels, shrink=True):
    """Within-Class Covariance Normalization (Hatch, Kajarekar & Stolcke 2006).

    Sinif-ici = AYNI KISININ farkli arac satirlari. W kestirilip X @ W^(-1/2).
    19 ozellik x ~13-16 kisi -> W tam ruteli DEGIL, Ledoit-Wolf shrinkage sart.
    """
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    dev = [X[labels == lab] - X[labels == lab].mean(axis=0)
           for lab in np.unique(labels) if (labels == lab).sum() >= 2]
    if not dev:
        return X
    D = np.vstack(dev)
    if len(D) < 2:
        return X
    W = (LedoitWolf(assume_centered=True).fit(D).covariance_ if shrink
         else np.cov(D, rowvar=False))
    ev, V = np.linalg.eigh(W)
    ev = np.maximum(ev, 1e-10)
    return X @ (V @ np.diag(ev ** -0.5) @ V.T)


def levene_by_track(df, metric, group=None):
    """v5 DUZELTME 3: pist-ici ARTIK varyans homojen mi?

    TDD'nin ORTA rejimi heteroskedastisite yuzunden basarisiz olmustu; gercek
    veri de heteroskedastik olabilir ve bu R tahminlerini bozar.
    """
    g = group or ID_COL
    d = df[[metric, 'track', g]].dropna().copy()
    if d['track'].nunique() < 2:
        return dict(stat=np.nan, p=np.nan, n_groups=0)
    d['_r'] = d[metric] - d.groupby(g)[metric].transform('mean')
    groups = [x['_r'].values for _, x in d.groupby('track') if len(x) >= 3]
    if len(groups) < 2:
        return dict(stat=np.nan, p=np.nan, n_groups=len(groups))
    s, p = _levene(*groups, center='median')
    return dict(stat=float(s), p=float(p), n_groups=len(groups))


def _wccn_transform(A, B, pairs):
    """Verilen (i, j) ciftlerinden sinif-ici kovaryans -> W^(-1/2) matrisi."""
    dev = []
    for i, j in pairs:
        P = np.vstack([A[i], B[j]])
        dev.append(P - P.mean(axis=0))
    D = np.vstack(dev)
    W = LedoitWolf(assume_centered=True).fit(D).covariance_
    ev, V = np.linalg.eigh(W)
    ev = np.maximum(ev, 1e-10)
    return V @ np.diag(ev ** -0.5) @ V.T


def _corr_sim_row(a, Bm):
    a = a - a.mean()
    Bc = Bm - Bm.mean(axis=1, keepdims=True)
    return (Bc @ a) / (np.linalg.norm(a) * np.linalg.norm(Bc, axis=1) + 1e-12)


def _loo_wccn_accuracy(A, B, perm):
    """Kisi i eslestirilirken W, DIGER kisilerin ciftlerinden kestirilir."""
    n = len(A); hits = 0
    for i in range(n):
        T = _wccn_transform(A, B, [(j, perm[j]) for j in range(n) if j != i])
        hits += int(np.argmax(_corr_sim_row(A[i] @ T, B @ T)) == perm[i])
    return hits / n


def identifiability_loo_wccn(A, B, n_perm=2000, seed=RNG_SEED):
    """v5.1: WCCN'i SIZDIRMADAN uygulayan kimliklendirme.

    NEDEN GEREKTI — olculdu (n=12, 19 metrik, SINYALSIZ veri):
      naif WCCN (cift'lerin kendisinden kestirim): ort acc=0.542, YANLIS-POZITIF %88-97
      LOO-WCCN (kisi disarida birakilarak)       : ort acc=0.108, YANLIS-POZITIF %3.3
    Naif surumde permutasyon testi de gecersizdir: null dagilim ayni sizdirilmis
    uzayda hesaplanir. LOO'da whitening her permutasyon icin yeniden kurulur.
    """
    n = len(A)
    out = dict(n=n, accuracy=np.nan, chance=(1.0 / n if n else np.nan),
               idiff=np.nan, p_perm=np.nan, null=None, sim_method='loo_wccn',
               k_metrics=int(A.shape[1]), n_needed=np.nan)
    if n < 3:
        return out
    ident = np.arange(n)
    obs = _loo_wccn_accuracy(A, B, ident)
    T_all = _wccn_transform(A, B, [(j, j) for j in range(n)])
    S = np.array([_corr_sim_row(A[i] @ T_all, B @ T_all) for i in range(n)])
    rg = np.random.default_rng(seed)
    null = np.array([_loo_wccn_accuracy(A, B, rg.permutation(n)) for _ in range(n_perm)])
    out.update(accuracy=float(obs),
               idiff=float((np.diag(S).mean() - S[~np.eye(n, dtype=bool)].mean()) * 100),
               p_perm=float(((null >= obs).sum() + 1) / (n_perm + 1)),
               null=null, acc_needed=float(np.percentile(null, 95)))
    return out


def calibrate_identifiability(n, k, n_rep=40, n_perm=120, seed=0, mode='loo'):
    """Kendi olcum aracimizi SINYALSIZ veriyle kalibre eder.

    Beklenen: yanlis-pozitif orani ~ALPHA. Belirgin sapma varsa yontem gecersizdir.
    """
    rg = np.random.default_rng(seed); ps = []
    for _ in range(n_rep):
        A = rg.normal(0, 1, (n, k)); B = rg.normal(0, 1, (n, k))
        Z = zscore_cols(np.vstack([A, B])); A2, B2 = Z[:n], Z[n:]
        if mode == 'loo':
            r = identifiability_loo_wccn(A2, B2, n_perm=n_perm, seed=int(rg.integers(1e6)))
        else:
            T = _wccn_transform(A2, B2, [(j, j) for j in range(n)])
            r = identifiability(A2 @ T, B2 @ T, n_perm=n_perm, method='corr')
        ps.append(r['p_perm'])
    ps = np.asarray(ps)
    return dict(mode=mode, n=n, k=k, n_rep=n_rep,
                fpr=float(np.mean(ps < ALPHA)), median_p=float(np.median(ps)))


def zscore_cols(M):
    """Kolon bazinda z-skor; sabit kolonlar 0'a set edilir."""
    mu = np.nanmean(M, axis=0)
    sd = np.nanstd(M, axis=0)
    sd = np.where(sd < 1e-12, 1.0, sd)
    Z = (M - mu) / sd
    return np.nan_to_num(Z, nan=0.0)


SIM_METHOD_DEFAULT = 'corr'
K_MIN_FOR_CORR = 8      # k<8 ise korelasyon guvenilmez -> negeuc kullan

def similarity_matrix(A, B, method=None):
    """Profil benzerlik matrisi.

    method='corr'   : satir-bazli Pearson. k (metrik sayisi) buyukse iyi.
    method='negeuc' : negatif standartlastirilmis Oklid mesafesi. Her k'de calisir.

    NEDEN IKI OLCUT: v1'de boyut duzeyi analizi k=4 metrikle korelasyon
    kullaniyordu; 4 noktadan korelasyon asiri gurultulu ve ic tutarsiz sonuc
    verdi (B3: Idiff negatif ama p anlamli). k<K_MIN_FOR_CORR ise otomatik
    negeuc'e gecilir.
    """
    k = A.shape[1]
    if method is None:
        method = 'corr' if k >= K_MIN_FOR_CORR else 'negeuc'
    if method == 'corr':
        Az = A - A.mean(axis=1, keepdims=True)
        Bz = B - B.mean(axis=1, keepdims=True)
        An = np.linalg.norm(Az, axis=1, keepdims=True); An[An < 1e-12] = 1.0
        Bn = np.linalg.norm(Bz, axis=1, keepdims=True); Bn[Bn < 1e-12] = 1.0
        return (Az / An) @ (Bz / Bn).T
    # negeuc: mesafeyi [-1, 1] araligina esle (Idiff yorumlanabilir kalsin)
    D = np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2))
    rng_ = D.max() - D.min()
    if rng_ < 1e-12:
        return np.zeros_like(D)
    return 1.0 - 2.0 * (D - D.min()) / rng_


def identifiability(A, B, n_perm=N_PERM, seed=RNG_SEED, method=None):
    """Finn vd. 2015 paradigmasi. A ve B ayni sirali surucu profilleri (n x p).

    Doner: accuracy (iki yon ortalamasi), Idiff, permutasyon p, null dagilim.
    """
    n = A.shape[0]
    used = method if method is not None else ('corr' if A.shape[1] >= K_MIN_FOR_CORR else 'negeuc')
    out = dict(n=n, accuracy=np.nan, acc_ab=np.nan, acc_ba=np.nan,
               idiff=np.nan, p_perm=np.nan, n_correct=0, chance=np.nan,
               null=None, sim_method=used, k_metrics=int(A.shape[1]),
               n_needed=np.nan)
    if n < 3:
        return out
    S = similarity_matrix(A, B, method=used)
    hit_ab = (S.argmax(axis=1) == np.arange(n))
    hit_ba = (S.argmax(axis=0) == np.arange(n))
    acc_ab, acc_ba = hit_ab.mean(), hit_ba.mean()
    diag = np.diag(S)
    off  = S[~np.eye(n, dtype=bool)]
    out.update(accuracy=float((acc_ab + acc_ba) / 2),
               acc_ab=float(acc_ab), acc_ba=float(acc_ba),
               idiff=float((diag.mean() - off.mean()) * 100),
               n_correct=int(hit_ab.sum() + hit_ba.sum()),
               chance=1.0 / n)
    rg = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        pi = rg.permutation(n)
        Sp = S[:, pi]
        null[i] = ((Sp.argmax(axis=1) == np.arange(n)).mean() +
                   (Sp.argmax(axis=0) == np.arange(n)).mean()) / 2
    out['p_perm'] = float(((null >= out['accuracy']).sum() + 1) / (n_perm + 1))
    out['null'] = null
    # Guc: p<0.05 icin gereken dogruluk (null dagilimin %95 kuantili)
    out['acc_needed'] = float(np.percentile(null, 95))
    out['n_needed'] = float(np.ceil(out['acc_needed'] * n))
    return out


_lmm_errors = []

def repeatability_lmm(df, metric, group=None, fixed=('track', 'car')):
    group = group or ID_COL   # v5: kimlik birimi KISI
    """Adjusted repeatability R = s2_group / (s2_group + s2_resid).

    statsmodels varsa MixedLM; yoksa tek-yonlu ANOVA bileseni (fallback).
    """
    d = df[[group, metric] + [f for f in fixed if f in df.columns]].dropna()
    if d[group].nunique() < 3 or len(d) < 6:
        return dict(R=np.nan, var_g=np.nan, var_e=np.nan, method='yetersiz_n', n=len(d), converged=None, boundary=False)
    if d[metric].std() < 1e-12:
        return dict(R=np.nan, var_g=np.nan, var_e=np.nan, method='sabit_metrik', n=len(d), converged=None, boundary=False)

    if HAS_SM:
        rhs = " + ".join(f"C({f})" for f in fixed
                         if f in d.columns and d[f].nunique() > 1)
        formula = f"Q('{metric}') ~ " + (rhs if rhs else "1")
        try:
            # Yerel uyari baglami: global filtreler Jupyter'da sifirlanabiliyor.
            # ConvergenceWarning burada BEKLENEN bir durum (R~0 metriklerde sinir vakasi),
            # bastiriliyor ama sonuc 'yakinsadi' sutununda RAPORLANIYOR.
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                # v5.1 DUZELTME: 'lbfgs' analitik gradyanda xtvix'i tersliyor;
                # seyrek/dengesiz tasarimda TEKIL matris -> ya LinAlgError ya
                # sinir cozumu (vg=0, sahte R=0). Olculdu (18 kisi, sentetik):
                #   lbfgs  NULL 8/8 hata · ORTA 7/8 hata · GUCLU 0.768
                #   powell NULL 0.082    · ORTA 0.291    · GUCLU 0.768
                # powell turevsizdir, matris tersleme yok.
                try:
                    m = smf.mixedlm(formula, d, groups=d[group]).fit(reml=True, method='powell')
                except Exception:
                    m = smf.mixedlm(formula, d, groups=d[group]).fit(reml=True, method='lbfgs')
            vg = float(np.asarray(m.cov_re)[0, 0]); ve = float(m.scale)
            if not np.isfinite(vg) or not np.isfinite(ve) or (vg + ve) <= 0:
                raise ValueError("gecersiz varyans")
            conv = bool(getattr(m, 'converged', False))
            # Sinir vakasi: varyans bilesenlerinden biri fiilen sifir
            boundary = (vg < 1e-10) or (ve < 1e-10)
            return dict(R=vg / (vg + ve), var_g=vg, var_e=ve,
                        method='mixedlm' if conv else 'mixedlm_YAKINSAMADI',
                        n=len(d), converged=conv, boundary=bool(boundary))
        except Exception as _e:
            _lmm_errors.append((metric, str(_e)[:80]))

    # --- Fallback: tek-yonlu rastgele etki ANOVA'si (pist/arac KONTROLSUZ) ---
    g = d.groupby(group)[metric]
    ni = g.size().values; mi = g.mean().values
    k = len(ni); N = ni.sum()
    if k < 2:
        return dict(R=np.nan, var_g=np.nan, var_e=np.nan, method='yetersiz_grup', n=len(d), converged=None, boundary=False)
    grand = d[metric].mean()
    msb = ((ni * (mi - grand) ** 2).sum()) / (k - 1)
    ssw = sum(((d[d[group] == gid][metric] - mu) ** 2).sum()
              for gid, mu in zip(g.groups.keys(), mi))
    msw = ssw / (N - k) if N > k else np.nan
    n0 = (N - (ni ** 2).sum() / N) / (k - 1)
    vg = max((msb - msw) / n0, 0.0) if np.isfinite(msw) and n0 > 0 else np.nan
    ve = msw
    R = vg / (vg + ve) if np.isfinite(vg) and np.isfinite(ve) and (vg + ve) > 0 else np.nan
    return dict(R=R, var_g=vg, var_e=ve, method='anova_fallback', n=len(d),
                converged=None, boundary=bool(np.isfinite(vg) and vg < 1e-10))


def bootstrap_R(df, metric, n_boot=N_BOOT, seed=RNG_SEED):
    """Surucu duzeyinde yeniden ornekleme ile R icin %95 GA."""
    # v5 duzeltme: kimlik birimi ID_COL. driver_id uzerinden orneklenip
    # person ile gruplanirsa bootstrap kopyalari AYNI person etiketini paylasir
    # ve varyans coker -> GA'lar anlamsiz olur.
    ids = df[ID_COL].unique()
    rg = np.random.default_rng(seed)
    vals = []
    _wctx = warnings.catch_warnings(); _wctx.__enter__(); warnings.simplefilter("ignore")
    for _ in range(n_boot):
        samp = rg.choice(ids, len(ids), replace=True)
        parts = []
        for j, s in enumerate(samp):
            t = df[df[ID_COL] == s].copy()
            t[ID_COL] = f"B{j:03d}"
            parts.append(t)
        bd = pd.concat(parts, ignore_index=True)
        r = repeatability_lmm(bd, metric)['R']
        if np.isfinite(r):
            vals.append(r)
    _wctx.__exit__(None, None, None)
    if len(vals) < 20:
        return (np.nan, np.nan, len(vals))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return (float(lo), float(hi), len(vals))


def bh_fdr(pvals, alpha=ALPHA):
    """Benjamini-Hochberg. Doner: (reddedildi_mi dizisi, duzeltilmis p)."""
    p = np.asarray(pvals, float)
    ok = np.isfinite(p)
    q = np.full_like(p, np.nan)
    rej = np.zeros(len(p), bool)
    if ok.sum() == 0:
        return rej, q
    idx = np.where(ok)[0]
    order = idx[np.argsort(p[idx])]
    m = len(order)
    prev = 1.0
    for rank in range(m - 1, -1, -1):
        i = order[rank]
        val = min(prev, p[i] * m / (rank + 1))
        q[i] = val
        prev = val
    rej[ok] = q[ok] <= alpha
    return rej, q


def cv_pct(vals, ddof=0):
    """NB14 ile ayni betimsel CV (ham degerler, ddof=0)."""
    v = np.asarray([x for x in vals if np.isfinite(x)], float)
    if len(v) < 2 or abs(v.mean()) < 1e-12:
        return np.nan
    return 100.0 * v.std(ddof=ddof) / abs(v.mean())


print("Tahmin ediciler yuklendi: identifiability · repeatability_lmm · bootstrap_R · bh_fdr · cv_pct")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 2b — TDD dogrulamasi (sentetik, bilinen cevap)
# ══════════════════════════════════════════════════════════════
def make_synth(regime, n_drivers=18, seed=7):
    """Efe'nin tasarimini taklit eder: 18 surucu, 2-3 arac, 3 pist, dengesiz."""
    rg = np.random.default_rng(seed)
    sd_driver = {'NULL': 0.0, 'ORTA': 1.0, 'GUCLU': 3.0}[regime]
    rows = []
    for d in range(n_drivers):
        eff = rg.normal(0, 1, size=len(ALL_METRICS)) * sd_driver
        cars = CARS if d < 5 else CARS[:2]
        for c in cars:
            car_eff = rg.normal(0, 0.6, size=len(ALL_METRICS)) * (CARS.index(c) + 1) / 3
            for t in TRACKS:
                if rg.random() < 0.20:
                    continue
                trk = rg.normal(0, 1.5, size=len(ALL_METRICS)) * (TRACKS.index(t) + 1) / 3
                noise = rg.normal(0, 1.0, size=len(ALL_METRICS))
                # v5 duzeltme: ID_COL="person" oldugu icin sentetik veri de
                # person kolonu tasimali — TDD gercek veriyle AYNI yolu test eder.
                row = dict(driver_id=f"S{d:02d}", person=f"S{d:02d}", car=c, track=t)
                for j, m in enumerate(ALL_METRICS):
                    row[m] = 10 + eff[j] + car_eff[j] + trk[j] + noise[j]
                rows.append(row)
    return pd.DataFrame(rows)


def quick_ident(df, car_a, car_b, metrics=ALL_METRICS, n_perm=2000):
    """Pist-artik uzerinden havuzlanmis identifiability (TDD icin hizli surum)."""
    d = df.copy()
    for m in metrics:                      # pist etkisini cikar
        d[m] = d[m] - d.groupby('track')[m].transform('mean')
    prof = d.groupby(['driver_id', 'car'])[metrics].mean().reset_index()
    a = prof[prof.car == car_a].set_index('driver_id')
    b = prof[prof.car == car_b].set_index('driver_id')
    common = sorted(set(a.index) & set(b.index))
    if len(common) < 3:
        return dict(n=len(common), accuracy=np.nan, p_perm=np.nan, idiff=np.nan)
    Z = zscore_cols(pd.concat([a.loc[common], b.loc[common]])[metrics].values)
    n = len(common)
    return identifiability(Z[:n], Z[n:], n_perm=n_perm)


print("=" * 68)
print("  TDD — sentetik veri ile tahmin edici dogrulamasi")
print("=" * 68)
_tdd_rows = []
for regime, exp_ident, exp_R in [('NULL', 'sansa yakin', '~0.0'),
                                 ('ORTA', 'sanstan yuksek', '~0.5'),
                                 ('GUCLU', 'cok yuksek', '~0.9')]:
    sdf = make_synth(regime)
    idr = quick_ident(sdf, CARS[0], CARS[1])
    rr  = [repeatability_lmm(sdf, m)['R'] for m in ALL_METRICS[:6]]
    rr  = [x for x in rr if np.isfinite(x)]
    medR = float(np.median(rr)) if rr else np.nan
    _tdd_rows.append(dict(rejim=regime, n=idr['n'],
                          accuracy=idr['accuracy'], p=idr['p_perm'],
                          idiff=idr['idiff'], medyan_R=medR,
                          beklenen_ident=exp_ident, beklenen_R=exp_R))
    print(f"  {regime:<6} n={idr['n']:>2} | accuracy={idr['accuracy']:.3f} "
          f"(sans={1/max(idr['n'],1):.3f}) p={idr['p_perm']:.4f} "
          f"Idiff={idr['idiff']:>6.2f} | medyan R={medR:.3f}  "
          f"[beklenen: {exp_ident} / {exp_R}]")

tdd_df = pd.DataFrame(_tdd_rows)
_ok_null   = (tdd_df.loc[tdd_df.rejim == 'NULL',  'p'].iloc[0] > 0.05)
_ok_guclu  = (tdd_df.loc[tdd_df.rejim == 'GUCLU', 'p'].iloc[0] < 0.05)
_ok_mono   = (tdd_df.loc[tdd_df.rejim == 'NULL', 'medyan_R'].iloc[0]
              < tdd_df.loc[tdd_df.rejim == 'GUCLU', 'medyan_R'].iloc[0])
print()
print(f"  [{'GECTI' if _ok_null  else 'KALDI'}] NULL rejiminde identifiability anlamsiz")
print(f"  [{'GECTI' if _ok_guclu else 'KALDI'}] GUCLU rejiminde identifiability anlamli")
print(f"  [{'GECTI' if _ok_mono  else 'KALDI'}] R, NULL -> GUCLU yonunde artiyor")
# ── v5.1 KAPI 0: KIMLIKLENDIRME ARACININ KENDI KALIBRASYONU ──
# Sinyalsiz veriyle yanlis-pozitif orani ~ALPHA olmali. v5'te naif WCCN
# %88-97 yanlis pozitif uretiyordu ve bu ancak KOSUDAN SONRA fark edildi.
CALIB = [calibrate_identifiability(n=12, k=len(ALL_METRICS), n_rep=30, n_perm=100,
                                   seed=11, mode=m) for m in ('loo', 'naive')]
for _c in CALIB:
    _flag = 'GECTI' if _c['fpr'] <= 0.15 else 'KALDI — YONTEM GECERSIZ'
    print(f"  kalibrasyon [{_c['mode']:<5}] n={_c['n']} k={_c['k']}  "
          f"yanlis-pozitif={_c['fpr']:.3f} (nominal {ALPHA})  medyan p={_c['median_p']:.3f}  {_flag}")
CALIB_OK = bool([c for c in CALIB if c['mode'] == 'loo'][0]['fpr'] <= 0.15)
if not CALIB_OK:
    print("  !!! LOO-WCCN kalibrasyonu KALDI — kimliklendirme sonuclari GECERSIZ.")

# v5 DUZELTME 2: ORTA rejim kapisi.
# v4'te kapi yalnizca NULL < GUCLU kontrol ediyordu; ORTA rejim medyan R=0.000
# donduruyordu ve kimse fark etmedi. Deger zaten tdd_df'te var, yeniden hesaplamiyoruz.
_mid_med = float(tdd_df.loc[tdd_df.rejim == 'ORTA', 'medyan_R'].iloc[0])
_ok_orta = bool(np.isfinite(_mid_med) and 0.20 < _mid_med < 0.80)
print(f"  ORTA rejim  medyan R = {_mid_med:.3f}   [beklenen 0.20-0.80]  "
      f"{'GECTI' if _ok_orta else 'KALDI'}")
if not _ok_orta:
    print("  !!! ORTA kapisi KALDI — tahmin edici orta duzey sinyali yakalayamiyor.")
    print("      Levene ile pist-ici varyans homojenligini kontrol et;")
    print("      heteroskedastisite R'yi sifira cekiyor olabilir.")

TDD_PASS = bool(CALIB_OK and _ok_orta and _ok_null and _ok_guclu and _ok_mono)
print(f"\n  TDD SONUC: {'TUM TESTLER GECTI' if TDD_PASS else 'UYARI — bir test kaldi, ciktilari dikkatle oku'}")

## BLOK 3 — Veri: `(driver_id, car, track)` matrisi

**Neden yeniden segmentasyon:** mevcut `driver_corner_matrix_*.parquet` dosyalari yalnizca
`driver_id`'ye gore grupluyor; `car` kolonu dusuruluyor. Araclar arasi karsilastirma icin
`(driver_id, car, track)` kirilimi gerekiyor ve diskte yok. Segmentasyon cekirdegi
NB11 v2 ile **birebir ayni** — sonuclarin kademe hattiyla karsilastirilabilir kalmasi icin.

Beklenen is yuku: ~110 (surucu x arac x pist) kombinasyonu.

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 3a — NB08 v4 segmentasyon cekirdegi (NB11 v2 ile birebir ayni)
# ══════════════════════════════════════════════════════════════
SEARCH_BACK_M      = 300
SEARCH_FWD_M       = 200
TRAIL_BRAKE_ZONE   = 0.40
TRAIL_MIN_PRESSURE = 0.03
MIN_APEX_SPEED     = 20.0
MIN_ENTRY_SPEED    = 50.0
MC_RADIUS_M        = 15
EXIT_FALLBACK_M    = 50

def detect_steer_column(df):
    cols = {c: c.lower() for c in df.columns}
    for c, cl in cols.items():
        if any(k in cl for k in ['steerangle', 'steering_angle', 'steer_angle', 'wheel_angle']):
            return ('steer', c)
    for c, cl in cols.items():
        if 'steer' in cl and 'error' not in cl:
            return ('steer', c)
    for c, cl in cols.items():
        if any(k in cl for k in ['g_lat', 'glat', 'lateral_g', 'accg_y', 'accel_lat']):
            return ('g_lat', c)
    return ('speed_proxy', None)

def votes_to_confidence(n_votes):
    if n_votes >= 3: return 1.0
    elif n_votes == 2: return 0.7
    elif n_votes == 1: return 0.4
    return 0.0

def _empty_seg(apex_dist, apex_speed):
    return {'seg_apex_dist': apex_dist, 'seg_apex_speed': apex_speed,
            'seg_entry_dist': np.nan, 'seg_entry_speed': np.nan,
            'seg_exit_dist': np.nan, 'seg_exit_speed': np.nan,
            'seg_braking_dist': np.nan, 'seg_speed_loss_eff': np.nan,
            'seg_coasting_dist': np.nan, 'seg_trail_braking': False,
            'seg_trail_pressure': np.nan, 'seg_avg_brake_pressure': np.nan,
            'seg_entry_valid': False, 'seg_exit_method': 'none'}

def segment_corner(df_lap, apex_lapdist, difficulty):
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values   if 'accStatus'   in df_lap.columns else np.zeros(len(df_lap))
    dist  = df_lap['LapDist'].values

    apex_idx = int(np.argmin(np.abs(dist - apex_lapdist)))
    apex_speed = float(speed[apex_idx])

    back_limit = max(0, apex_idx - int(SEARCH_BACK_M / 2))
    entry_idx  = apex_idx
    for j in range(apex_idx - 1, back_limit, -1):
        if brake[j] < 0.05 and speed[j] > speed[apex_idx]:
            entry_idx = j
            break

    entry_speed = float(speed[entry_idx])
    if entry_speed < MIN_ENTRY_SPEED or apex_speed < MIN_APEX_SPEED:
        return _empty_seg(apex_lapdist, apex_speed)

    fwd_limit = min(len(speed) - 1, apex_idx + int(SEARCH_FWD_M / 2))
    exit_idx, exit_method = apex_idx, 'none'
    for j in range(apex_idx + 1, fwd_limit):
        if acc[j] > 0.3 and speed[j] > apex_speed:
            exit_idx = j; exit_method = 'throttle_full'; break
    if exit_idx == apex_idx:
        for j in range(apex_idx + 1, fwd_limit):
            if acc[j] > 0.3:
                exit_idx = j; exit_method = 'throttle_only'; break
    if exit_idx == apex_idx:
        fb_target = dist[apex_idx] + EXIT_FALLBACK_M
        fb_idx = int(np.argmin(np.abs(dist - fb_target)))
        exit_idx = min(fb_idx, fwd_limit); exit_method = 'distance_fallback'

    exit_speed   = float(speed[exit_idx])
    braking_dist = float(dist[apex_idx] - dist[entry_idx]) if entry_idx < apex_idx else 0.0
    speed_loss_eff = (apex_speed / entry_speed) if entry_speed > 0 else 0.0

    coast_start = apex_idx
    for j in range(apex_idx, fwd_limit):
        if brake[j] < 0.05:
            coast_start = j; break
    coast_end = coast_start
    for j in range(coast_start, fwd_limit):
        if acc[j] > 0.1:
            coast_end = j; break
    coasting_dist = float(dist[coast_end] - dist[coast_start]) if coast_end > coast_start else 0.0

    trail_braking, trail_pressure, avg_brake = False, 0.0, 0.0
    if braking_dist > 10:
        tz = entry_idx + int((apex_idx - entry_idx) * (1 - TRAIL_BRAKE_ZONE))
        tp = brake[tz:apex_idx]
        trail_braking  = bool(tp.mean() > TRAIL_MIN_PRESSURE) if len(tp) > 0 else False
        trail_pressure = float(tp.mean()) if len(tp) > 0 else 0.0
        bz = brake[entry_idx:apex_idx]
        avg_brake = float(bz.mean()) if len(bz) > 0 else 0.0

    return {'seg_apex_dist': apex_lapdist, 'seg_apex_speed': apex_speed,
            'seg_entry_dist': float(dist[entry_idx]), 'seg_entry_speed': entry_speed,
            'seg_exit_dist': float(dist[exit_idx]), 'seg_exit_speed': exit_speed,
            'seg_braking_dist': braking_dist, 'seg_speed_loss_eff': speed_loss_eff,
            'seg_coasting_dist': coasting_dist, 'seg_trail_braking': trail_braking,
            'seg_trail_pressure': trail_pressure, 'seg_avg_brake_pressure': avg_brake,
            'seg_entry_valid': True, 'seg_exit_method': exit_method}

def segment_corner_phases(df_lap, seg, steer_info):
    mode, steer_col = steer_info
    if not seg.get('seg_entry_valid', False) or pd.isna(seg.get('seg_entry_dist')):
        return {'phase_slb_dist': np.nan, 'phase_slb_decel_rate': np.nan,
                'phase_ce_dist': np.nan, 'phase_ce_brake_at_turnin': np.nan,
                'phase_mc_speed_ratio': np.nan, 'phase_mc_lateral_signal': np.nan,
                'phase_cex_throttle_lag': np.nan, 'phase_cex_accel_rate': np.nan,
                'phase_turnin_method': 'skipped'}

    dist  = df_lap['LapDist'].values
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values if 'accStatus' in df_lap.columns else np.zeros(len(df_lap))

    entry_idx = int(np.argmin(np.abs(dist - seg['seg_entry_dist'])))
    apex_idx  = int(np.argmin(np.abs(dist - seg['seg_apex_dist'])))
    exit_idx  = int(np.argmin(np.abs(dist - seg['seg_exit_dist'])))

    turnin_idx = entry_idx + int((apex_idx - entry_idx) * 0.4)
    turnin_method = 'speed_proxy_40pct'
    if mode == 'steer' and steer_col in df_lap.columns:
        steer = np.abs(df_lap[steer_col].values)
        region = steer[entry_idx:apex_idx]
        if len(region) > 5:
            wlen = min(11, len(region))
            if wlen % 2 == 0: wlen -= 1
            smoothed = savgol_filter(region, wlen, min(2, wlen - 1)) if wlen >= 3 else region
            threshold = smoothed.max() * 0.15
            for k, val in enumerate(smoothed):
                if val > threshold:
                    turnin_idx = entry_idx + k; turnin_method = 'steer_threshold'; break
    elif mode == 'g_lat' and steer_col in df_lap.columns:
        glat = np.abs(df_lap[steer_col].values)
        region = glat[entry_idx:apex_idx]
        if len(region) > 5:
            threshold = region.max() * 0.20
            for k, val in enumerate(region):
                if val > threshold:
                    turnin_idx = entry_idx + k; turnin_method = 'g_lat_threshold'; break
    turnin_idx = max(entry_idx + 1, min(turnin_idx, apex_idx - 1))

    mc_start = int(np.argmin(np.abs(dist - (seg['seg_apex_dist'] - MC_RADIUS_M))))
    mc_end   = int(np.argmin(np.abs(dist - (seg['seg_apex_dist'] + MC_RADIUS_M))))
    mc_start = max(mc_start, turnin_idx); mc_end = min(mc_end, exit_idx)

    slb_dist = float(dist[turnin_idx] - dist[entry_idx]) if turnin_idx > entry_idx else 0.0
    slb_decel_rate = (float(speed[entry_idx] - speed[turnin_idx]) / slb_dist) if slb_dist > 5 else 0.0
    ce_dist = float(dist[mc_start] - dist[turnin_idx]) if mc_start > turnin_idx else 0.0
    ce_brake_at_turnin = float(brake[turnin_idx]) if turnin_idx < len(brake) else 0.0

    mc_speed = float(speed[mc_start:mc_end+1].mean()) if mc_end > mc_start else float(speed[apex_idx])
    esv = float(speed[entry_idx])
    mc_speed_ratio = mc_speed / esv if esv > 0 else 0.0
    mc_lateral = 0.0
    if mode in ('steer', 'g_lat') and steer_col and steer_col in df_lap.columns:
        lv = np.abs(df_lap[steer_col].values[mc_start:mc_end+1])
        mc_lateral = float(lv.mean()) if len(lv) > 0 else 0.0

    cex_throttle_lag, cex_accel_rate = 0.0, 0.0
    if exit_idx > mc_end:
        for j in range(mc_end, exit_idx):
            if acc[j] > 0.1:
                cex_throttle_lag = float(dist[j] - dist[mc_end]); break
        cd = float(dist[exit_idx] - dist[mc_end])
        cex_accel_rate = (float(speed[exit_idx] - speed[mc_end]) / cd) if cd > 5 else 0.0

    return {'phase_slb_dist': slb_dist, 'phase_slb_decel_rate': slb_decel_rate,
            'phase_ce_dist': ce_dist, 'phase_ce_brake_at_turnin': ce_brake_at_turnin,
            'phase_mc_speed_ratio': mc_speed_ratio, 'phase_mc_lateral_signal': mc_lateral,
            'phase_cex_throttle_lag': cex_throttle_lag, 'phase_cex_accel_rate': cex_accel_rate,
            'phase_turnin_method': turnin_method}

def enrich_corners(corners_v3_df, df_driver, steer_info):
    segs = []
    for _, row in corners_v3_df.iterrows():
        seg = segment_corner(df_driver, float(row['apex_dist']),
                             float(row.get('difficulty_score', 0.5)))
        seg.update(segment_corner_phases(df_driver, seg, steer_info))
        segs.append(seg)
    e = corners_v3_df.reset_index(drop=True).join(pd.DataFrame(segs))
    e['confidence'] = e['n_votes'].apply(votes_to_confidence)
    e.loc[e['seg_entry_dist'].isna(), 'confidence'] = 0.0
    e.loc[e['seg_apex_speed'] < MIN_APEX_SPEED, 'confidence'] = 0.0
    e.loc[e['seg_entry_speed'] < e['seg_apex_speed'], 'confidence'] = 0.0
    return e

def wavg(values, weights):
    v = pd.to_numeric(pd.Series(values), errors='coerce').values
    w = np.asarray(weights, float)
    mask = ~np.isnan(v) & (w > 0)
    return float(np.average(v[mask], weights=w[mask])) if mask.sum() > 0 else np.nan

print("Segmentasyon cekirdegi hazir (NB11 v2 ile birebir).")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 3b — (driver_id, car, track) matrisini uret
# ══════════════════════════════════════════════════════════════
def build_metric_row(driver_id, car, track, enriched):
    w = enriched['confidence'].fillna(0).values
    valid = enriched[enriched['confidence'] > 0]
    if len(valid) == 0:
        return None
    rec = {'driver_id': driver_id, 'car': car, 'track': track,
           'n_corners_valid': len(valid),
           'mean_apex_speed':     wavg(enriched['seg_apex_speed'], w),
           'speed_loss_eff':      wavg(enriched['seg_speed_loss_eff'], w),
           'mean_brake_pressure': wavg(enriched['seg_avg_brake_pressure'], w),
           'mean_coasting_dist':  wavg(enriched['seg_coasting_dist'], w),
           'trail_braking_ratio': float(valid['seg_trail_braking'].mean()),
           'mean_trail_pressure': wavg(enriched['seg_trail_pressure'], w),
           'apex_speed_std':      float(valid['seg_apex_speed'].std())     if len(valid) > 1 else np.nan,
           'braking_dist_std':    float(valid['seg_braking_dist'].std())   if len(valid) > 1 else np.nan,
           'exit_speed_std':      float(valid['seg_exit_speed'].std())     if len(valid) > 1 else np.nan,
           'speed_loss_eff_std':  float(valid['seg_speed_loss_eff'].std()) if len(valid) > 1 else np.nan}
    # --- v5 S2: turetilmis ORANSAL metrikler ---
    # Hipotez: boyutsuz oranlar aracin mutlak olcek seviyesinden bagimsizdir.
    # Ipucu: mevcut en dusuk CV'li iki metrik (speed_loss_eff %1.6,
    # mean_mc_speed_ratio %2.6) zaten oran; en yuksekler (cex_accel_rate %90.9,
    # slb_dist %51.8) boyutlu.
    def _ratio(num_col, den_col, name, eps=1e-6):
        if num_col in enriched.columns and den_col in enriched.columns:
            num = pd.to_numeric(enriched[num_col], errors='coerce')
            den = pd.to_numeric(enriched[den_col], errors='coerce')
            ok = den.abs() > eps
            rec[name] = wavg((num / den).where(ok), w)
        else:
            rec[name] = np.nan

    _ratio('seg_exit_speed', 'seg_apex_speed', 'r_exit_over_apex')
    _ratio('phase_slb_dist', 'seg_braking_dist', 'r_slb_share')
    _ratio('seg_coasting_dist', 'corner_length', 'r_coast_share')

    # k_decel_use: v_entry^2 / (2 * slb_dist) — kinematik yavaslama.
    # Boyutsuz DEGIL; LOO-yuzdelik normalizasyonu hucre icinde olceksizlestirir.
    if 'seg_entry_speed' in enriched.columns and 'phase_slb_dist' in enriched.columns:
        _ve = pd.to_numeric(enriched['seg_entry_speed'], errors='coerce') / 3.6
        _sd = pd.to_numeric(enriched['phase_slb_dist'], errors='coerce')
        rec['k_decel_use'] = wavg((_ve ** 2 / (2 * _sd)).where(_sd > 1.0), w)
    else:
        rec['k_decel_use'] = np.nan

    # r_apex_position: minimum hizin viraj icindeki KONUMU.
    # GECERLILIK KAPISI (on-kayitli): corners_v3 geometrisinden geliyorsa bu
    # bir PIST SABITIDIR, kimlik tasimaz. Kapi S2 blogunda otomatik uygulanir.
    if all(c in enriched.columns for c in ('apex_dist', 'corner_start', 'corner_length')):
        _ad = pd.to_numeric(enriched['apex_dist'], errors='coerce')
        _cs = pd.to_numeric(enriched['corner_start'], errors='coerce')
        _cl = pd.to_numeric(enriched['corner_length'], errors='coerce')
        rec['r_apex_position'] = wavg(((_ad - _cs) / _cl).where(_cl > 1.0), w)
    else:
        rec['r_apex_position'] = np.nan

    if 'character_class' in valid.columns:
        for cc in ['heavy_braking', 'trail_braking', 'lift_coast', 'flat_out']:
            rec[f'pct_{cc}'] = float((valid['character_class'] == cc).mean())
    for out_col, src in {'mean_slb_dist': 'phase_slb_dist',
                         'mean_slb_decel': 'phase_slb_decel_rate',
                         'mean_ce_brake_turnin': 'phase_ce_brake_at_turnin',
                         'mean_mc_speed_ratio': 'phase_mc_speed_ratio',
                         'mean_mc_lateral': 'phase_mc_lateral_signal',
                         'mean_cex_throttle_lag': 'phase_cex_throttle_lag',
                         'mean_cex_accel_rate': 'phase_cex_accel_rate'}.items():
        if src in enriched.columns:
            rec[out_col] = wavg(enriched[src], w)
    return rec

t0 = time.time()
rows, n_combo, skipped = [], 0, []
for track in TRACKS:
    cpath = FEATURES_DIR / f"{track}_corners_v3.parquet"
    if not cpath.exists():
        skipped.append(f"{track}: corners_v3 yok"); continue
    corners_v3 = pd.read_parquet(cpath)
    for car in CARS:
        fpath = TIER_DIR / f"{track}_{car}_human.parquet"
        if not fpath.exists():
            skipped.append(f"{track}/{car}: dosya yok"); continue
        df = pd.read_parquet(fpath)
        if 'driver_id' not in df.columns:
            skipped.append(f"{track}/{car}: driver_id yok"); continue
        df = df[~df['driver_id'].isin(DEGENERATE_IDS)]
        req = ['speed_kmh', 'LapDist', 'brakeStatus', 'accStatus']
        if any(c not in df.columns for c in req):
            skipped.append(f"{track}/{car}: kolon eksik"); continue
        steer_info = detect_steer_column(df)
        for did in sorted(df['driver_id'].unique()):
            sub = df[df['driver_id'] == did].reset_index(drop=True)
            rec = build_metric_row(did, car, track, enrich_corners(corners_v3, sub, steer_info))
            n_combo += 1
            if rec is not None:
                rows.append(rec)
        print(f"  {track:<14} {car:<16} {df['driver_id'].nunique():>3} surucu "
              f"({time.time()-t0:>5.1f}s)")

M = pd.DataFrame(rows)

# v5: KISI kolonu — kimlik birimi artik oturum-gunu degil kisi
M['person'] = M['driver_id'].map(person_of)

# v5: turetilmis metriklerin gercekten uretilip uretilmedigini LOUD raporla
print("\n--- v5 turetilmis metrikler (S2) ---")
for _dm in DERIVED_METRICS:
    if _dm not in M.columns:
        M[_dm] = np.nan
    _ok = int(M[_dm].notna().sum())
    print(f"  {_dm:<20} gecerli satir: {_ok:>4}/{len(M)}"
          f"{'   !!! URETILMEDI — kaynak kolon yok' if _ok == 0 else ''}")
for m in ALL_METRICS:
    if m not in M.columns:
        M[m] = np.nan
        print(f"  UYARI: '{m}' uretilmedi -> NaN")

print(f"\nIslenen kombinasyon : {n_combo}")
print(f"Gecerli satir       : {len(M)}")
print(f"Benzersiz surucu    : {M['driver_id'].nunique()}")
print(f"Sure                : {time.time()-t0:.1f}s")
if skipped:
    print("Atlananlar:", skipped)

M.to_parquet(OUT_DIR / "driver_car_track_matrix.parquet", index=False)
print(f"Kaydedildi: {OUT_DIR / 'driver_car_track_matrix.parquet'}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 3c — Tani: tasarim dengesi + arac/pist cakismasi
# ══════════════════════════════════════════════════════════════
print("=" * 68)
print("  TASARIM TANISI")
print("=" * 68)

design = (M.groupby([ID_COL, 'car'])['track'].nunique()
            .unstack(fill_value=0))
for c in CARS:
    if c not in design.columns:
        design[c] = 0
design = design[CARS]
design['n_cars'] = (design > 0).sum(axis=1)
design = design.sort_values('n_cars', ascending=False)
print("\nSurucu x Arac (hucre = kac pist):")
print(design.to_string())
print("\nArac sayisina gore surucu dagilimi:")
print(design['n_cars'].value_counts().sort_index().to_string())

multi = design[design['n_cars'] >= 2].index.tolist()
tri   = design[design['n_cars'] >= 3].index.tolist()
print(f"\n>=2 arac kullanan surucu : {len(multi)}")
print(f"  3 arac kullanan surucu : {len(tri)}  -> {tri}")

# --- Arac x Pist cakisma kontrolu ---
print("\n" + "-" * 68)
print("  ARAC <-> PIST CAKISMASI (ayrisabiliyor mu?)")
print("-" * 68)
ct = pd.crosstab(M['car'], M['track'])
print(ct.to_string())
_zero = (ct == 0).sum().sum()
print(f"\nBos hucre: {_zero}/{ct.size}  -> "
      f"{'AYRISABILIR (her arac birden fazla pistte)' if _zero == 0 else 'DIKKAT: bazi arac-pist kombinasyonu yok'}")

# --- Pist-ici arac ciftleri (temiz senaryo icin n) ---
print("\n" + "-" * 68)
print("  PIST-ICI ARAC CIFTLERI (senaryo A icin kullanilabilir n)")
print("-" * 68)
pair_counts = {}
for track in sorted(M['track'].unique()):
    sub = M[M['track'] == track]
    for ca, cb in combinations(CARS, 2):
        sa = set(sub[sub.car == ca][ID_COL])
        sb = set(sub[sub.car == cb][ID_COL])
        n = len(sa & sb)
        pair_counts[(track, ca, cb)] = n
        flag = "guclu" if n >= 10 else ("orta" if n >= 6 else ("zayif" if n >= 3 else "YETERSIZ"))
        print(f"  {track:<14} {ca[:12]:<12} x {cb[:12]:<12}  n={n:>3}  [{flag}]")

# --- Pist-artik havuzlanmis (senaryo B icin n) ---
print("\n" + "-" * 68)
print("  PIST-ARTIK HAVUZLANMIS (senaryo B icin kullanilabilir n)")
print("-" * 68)
for ca, cb in combinations(CARS, 2):
    sa = set(M[M.car == ca][ID_COL]); sb = set(M[M.car == cb][ID_COL])
    n = len(sa & sb)
    flag = "guclu" if n >= 10 else ("orta" if n >= 6 else ("zayif" if n >= 3 else "YETERSIZ"))
    print(f"  {ca[:14]:<14} x {cb[:14]:<14}  n={n:>3}  [{flag}]")

# ── v5: KIMLIK MERDIVENI — hangi basamak kac kisiyle olculebilir? ──
print("\n" + "=" * 68)
print("  KIMLIK MERDIVENI (v5)")
print("=" * 68)
_sess = M.groupby(ID_COL)['driver_id'].nunique()
print(f"\nKisi sayisi                     : {M[ID_COL].nunique()}")
print(f"Oturum-gunu (driver_id) sayisi  : {M['driver_id'].nunique()}")
print(f"Birden fazla oturum-gunu olan   : {int((_sess >= 2).sum())}  "
      f"-> {sorted(_sess[_sess >= 2].index.tolist())}")

# R2: ayni ARAC, farkli GUN
_r2 = (M.groupby([ID_COL, 'car'])['driver_id'].nunique()
         .reset_index(name='n_sessions'))
_r2 = _r2[_r2['n_sessions'] >= 2]
print(f"\nBASAMAK 2 (gun-arasi, AYNI arac) uygun (kisi,arac) hucresi: {len(_r2)}")
if len(_r2):
    print(_r2.to_string(index=False))
    print(f"  -> {_r2[ID_COL].nunique()} farkli kisi")
else:
    print("  YOK — Basamak 2 olculemez")

# R3: coklu arac
_r3 = design[design['n_cars'] >= 2].index.tolist()
print(f"\nBASAMAK 3 (araclar-arasi) uygun kisi: {len(_r3)}")
print(f"BASAMAK 1 (gun-ici tur yarilama)    : {'ACIK' if DO_RUNG1 else 'KAPALI (DO_RUNG1=False)'}")

LADDER_INFO = dict(
    policy=POLICY,
    n_persons=int(M[ID_COL].nunique()),
    n_session_days=int(M['driver_id'].nunique()),
    persons_multi_session=sorted(_sess[_sess >= 2].index.tolist()),
    rung2_cells=[{'person': r[ID_COL], 'car': r['car'], 'n_sessions': int(r['n_sessions'])}
                 for _, r in _r2.iterrows()],
    rung2_n_persons=int(_r2[ID_COL].nunique()) if len(_r2) else 0,
    rung3_persons=sorted(_r3),
    rung1_enabled=bool(DO_RUNG1),
)

DESIGN_INFO = dict(
    n_rows=int(len(M)), n_drivers=int(M[ID_COL].nunique()),
    n_session_days=int(M['driver_id'].nunique()),
    n_multi_car=len(multi), n_three_car=len(tri), three_car_ids=tri,
    car_track_empty_cells=int(_zero),
    within_track_pairs={f"{t}|{a}|{b}": int(v) for (t, a, b), v in pair_counts.items()},
)

## BLOK 4 — Identifiability (birincil kanit)

Finn vd. 2015 paradigmasi: surucunun A aracindaki profilinden, B aracindaki profilini
bulabiliyor muyuz? Anlamlilik permutasyonla kurulur (kimlikler karistirilir, null dagilim).
Amico & Goni 2018'in **differential identifiability** olcutu de raporlanir:
birey-ici benzerlik ortalamasi eksi bireyler-arasi benzerlik ortalamasi.

**Iki senaryo yan yana:**
- **A — pist-ici:** her pist icin ayri; pist etkisi tamamen kontrol altinda, n kucuk
- **B — pist-artik havuzlanmis:** metrikten pist ortalamasi cikarilir, tum pistler havuzlanir; n buyuk

Ikisi ayni yonu gosterirse sonuc saglam; ayrisirsa bu da raporlanir.

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 4 — Identifiability: senaryo A (pist-ici) ve B (pist-artik)
# ══════════════════════════════════════════════════════════════
def profiles_for_pair(data, car_a, car_b, metrics, prep='loo_wccn'):
    """Ortak KISILER icin (A, B) profil matrisleri.

    v5 ON-KAYITLI ZINCIR (prep='loo_wccn'):
      1. metrik -> hucre (arac,pist) ici LEAVE-ONE-OUT yuzdeligi  [cagiran uygular]
      2. kisi profili = coklu oturum/pist HAVUZLAMASI (groupby ortalamasi)
      3. shrinkage-WCCN whitening (sinif-ici = ayni kisinin farkli arac satirlari)
      4. eslestirme: whitened nearest-prototype
    prep='zscore' v4 davranisidir; sureklilik icin karsilastirma olarak raporlanir.
    """
    a = data[data.car == car_a].groupby(ID_COL)[metrics].mean()
    b = data[data.car == car_b].groupby(ID_COL)[metrics].mean()
    common = sorted(set(a.index) & set(b.index))
    if len(common) < 3:
        return None, None, common
    A0, B0 = a.loc[common], b.loc[common]
    both = pd.concat([A0, B0])
    Z = zscore_cols(both[metrics].values)
    n = len(common)
    # v5.1: whitening ARTIK BURADA YAPILMIYOR — sizdiriyordu.
    # LOO-WCCN, identifiability_loo_wccn icinde her kisi icin yeniden kurulur.
    return Z[:n], Z[n:], common


def residualize_track(data, metrics):
    d = data.copy()
    for m in metrics:
        d[m] = d[m] - d.groupby('track')[m].transform('mean')
    return d


def run_identifiability(data, metrics, label, n_perm=N_PERM, min_n=3, prep='loo_wccn'):
    """Tum arac ciftleri icin identifiability. Doner: DataFrame + null dagilimlari."""
    res, nulls = [], {}
    for ca, cb in combinations(CARS, 2):
        A, B, common = profiles_for_pair(data, ca, cb, metrics, prep=prep)
        if A is None or len(common) < min_n:
            res.append(dict(senaryo=label, car_a=ca, car_b=cb, n=len(common),
                            accuracy=np.nan, chance=np.nan, idiff=np.nan,
                            p_perm=np.nan, anlamli=False))
            continue
        if prep == 'loo_wccn':
            r = identifiability_loo_wccn(A, B, n_perm=min(n_perm, N_PERM_LOO))
        else:
            r = identifiability(A, B, n_perm=n_perm, method='corr')
        r2 = identifiability(A, B, n_perm=n_perm, method='negeuc')
        nulls[f"{label}|{ca}|{cb}"] = r['null']
        res.append(dict(senaryo=label, on_islem=prep, car_a=ca, car_b=cb, n=r['n'],
                        accuracy=r['accuracy'], chance=r['chance'],
                        idiff=r['idiff'], p_perm=r['p_perm'],
                        n_gereken=r.get('n_needed', np.nan),
                        acc_euc=r2['accuracy'], p_euc=r2['p_perm'],
                        anlamli=bool(np.isfinite(r['p_perm']) and r['p_perm'] < ALPHA),
                        anlamli_euc=bool(np.isfinite(r2['p_perm']) and r2['p_perm'] < ALPHA)))
    return pd.DataFrame(res), nulls


print("=" * 68)
print("  SENARYO B — pist-artik havuzlanmis (buyuk n)")
print("=" * 68)
# v5: on-kayitli zincirin 1. adimi — LOO yuzdelik (pist artigi yerine)
print("\nOn-isleme: hucre (arac,pist) ici leave-one-out yuzdeligi...")
M_loo = loo_percentile(M, ALL_METRICS + DERIVED_METRICS, cell_cols=('car', 'track'))
M_res = residualize_track(M, ALL_METRICS)   # v4 karsilastirmasi icin korundu
ident_B, nulls_B = run_identifiability(M_loo, ALL_METRICS, 'B_pist_artik', prep='loo_wccn')
ident_B_v4, _ = run_identifiability(M_res, ALL_METRICS, 'B_v4_zskor', prep='zscore')
print("\n[v4 karsilastirmasi — pist-artik + z-skor, on-islemsiz]")
print(ident_B_v4[['car_a', 'car_b', 'n', 'accuracy', 'p_perm']].to_string(index=False))
print(ident_B.to_string(index=False))

print()
print("=" * 68)
print("  SENARYO A — pist-ici (temiz, kucuk n)")
print("=" * 68)
rows_A, nulls_A = [], {}
for track in sorted(M['track'].unique()):
    sub = M[M['track'] == track]
    dfA, nA = run_identifiability(sub, ALL_METRICS, f'A_{track}', n_perm=N_PERM)
    dfA.insert(1, 'track', track)
    rows_A.append(dfA); nulls_A.update(nA)
ident_A = pd.concat(rows_A, ignore_index=True) if rows_A else pd.DataFrame()
if len(ident_A):
    print(ident_A.to_string(index=False))

# --- Ozet karsilastirma ---
print()
print("-" * 68)
print("  SENARYO KARSILASTIRMASI")
print("-" * 68)
valid_B = ident_B[ident_B['n'] >= 3]
valid_A = ident_A[ident_A['n'] >= 3] if len(ident_A) else ident_A
def _summ(df, name):
    if len(df) == 0:
        print(f"  {name:<16} kullanilabilir cift yok"); return dict(n_pairs=0)
    sig = int(df['anlamli'].sum())
    print(f"  {name:<16} {len(df)} cift, {sig} anlamli | "
          f"ort accuracy={df['accuracy'].mean():.3f} (ort sans={df['chance'].mean():.3f}) | "
          f"ort Idiff={df['idiff'].mean():.2f}")
    return dict(n_pairs=int(len(df)), n_sig=sig,
                mean_acc=float(df['accuracy'].mean()),
                mean_chance=float(df['chance'].mean()),
                mean_idiff=float(df['idiff'].mean()))
sB = _summ(valid_B, "B (havuzlanmis)")
sA = _summ(valid_A, "A (pist-ici)")

IDENT_SIGNIFICANT = bool(sB.get('n_sig', 0) > 0 or sA.get('n_sig', 0) > 0)
print(f"\n  IDENTIFIABILITY ANLAMLI MI: {'EVET' if IDENT_SIGNIFICANT else 'HAYIR'}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 5 — Mekanizma: boyut bazli + leave-one-metric-out
# ══════════════════════════════════════════════════════════════
BEST = valid_B.sort_values('n', ascending=False).iloc[0] if len(valid_B) else None
if BEST is None:
    print("Yeterli arac cifti yok — mekanizma analizi atlandi.")
    dim_ident = pd.DataFrame(); lomo = pd.DataFrame()
else:
    CA, CB, NB_ = BEST['car_a'], BEST['car_b'], int(BEST['n'])
    print(f"En genis cift: {CA} x {CB} (n={NB_}) — mekanizma bu cift uzerinde\n")

    print("=" * 68); print("  BOYUT BAZLI IDENTIFIABILITY"); print("=" * 68)
    print("  NOT: boyut basina yalnizca 4-6 metrik var. Satir-bazli korelasyon")
    print("       bu kadar az noktada guvenilmez (v1'de B3 ic tutarsiz sonuc verdi:")
    print("       Idiff negatif ama p anlamli). Bu yuzden boyut duzeyinde")
    print("       NEGATIF OKLID benzerligi kullaniliyor. Karsilastirma icin")
    print("       korelasyon sonucu da yazdiriliyor.\n")
    rows = []
    for dim, mets in DIMENSIONS.items():
        A, B, common = profiles_for_pair(M_res, CA, CB, mets)
        if A is None:
            rows.append(dict(boyut=dim, n_metrik=len(mets), n=len(common),
                             accuracy=np.nan, idiff=np.nan, p_perm=np.nan)); continue
        r  = identifiability(A, B, n_perm=N_PERM, method='negeuc')
        rc = identifiability(A, B, n_perm=N_PERM, method='corr')
        tutarli = (r['idiff'] > 0) == (r['accuracy'] > r['chance'])
        rows.append(dict(boyut=dim, n_metrik=len(mets), n=r['n'],
                         accuracy=r['accuracy'], idiff=r['idiff'], p_perm=r['p_perm'],
                         acc_corr=rc['accuracy'], p_corr=rc['p_perm'],
                         ic_tutarli=bool(tutarli)))
    dim_ident = pd.DataFrame(rows)
    dim_ident['anlamli'] = dim_ident['p_perm'] < ALPHA
    print(dim_ident.to_string(index=False))

    print()
    print("=" * 68); print("  LEAVE-ONE-METRIC-OUT (hangi metrik kritik?)"); print("=" * 68)
    A0, B0, _ = profiles_for_pair(M_res, CA, CB, ALL_METRICS)
    base = identifiability(A0, B0, n_perm=2000)['accuracy']
    rows = []
    for m in ALL_METRICS:
        sub_m = [x for x in ALL_METRICS if x != m]
        A, B, _ = profiles_for_pair(M_res, CA, CB, sub_m)
        acc = identifiability(A, B, n_perm=2000)['accuracy'] if A is not None else np.nan
        rows.append(dict(cikarilan=m, boyut=METRIC2DIM[m], accuracy=acc,
                         delta=acc - base if np.isfinite(acc) else np.nan))
    lomo = pd.DataFrame(rows).sort_values('delta')
    print(f"  Tam set accuracy = {base:.3f}  (k=19 metrik, benzerlik=korelasyon)\n")
    print("  En cok DUSUREN (yani en kritik) 6 metrik:")
    print(lomo.head(6).to_string(index=False))
    print("\n  En cok YUKSELTEN (yani gurultu olan) 4 metrik:")
    print(lomo.tail(4).to_string(index=False))

## BLOK 6 — Repeatability (mekanizma capraz kontrolu)

Her metrik icin karisik model: `metrik ~ C(pist) + C(arac) + (1 | surucu)`.
**Adjusted repeatability** R = sigma2_surucu / (sigma2_surucu + sigma2_artik) — sabit
etkilerin varyansi paydadan dislanir (Nakagawa & Schielzeth 2010).

19 test icin Benjamini-Hochberg FDR uygulanir. Guven araliklari surucu duzeyinde
bootstrap ile. **On-kayitta beyan edilen hassasiyet sinirini hatirla:** R~0.5'te beklenen
GA genisligi ~0.64.

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 6 — Repeatability: 19 metrik x MixedLM + bootstrap + BH-FDR
# ══════════════════════════════════════════════════════════════
t0 = time.time()
# Uyari filtresini tazele — Jupyter hucreler arasi sifirlayabiliyor.
warnings.filterwarnings('ignore')
if HAS_SM:
    from statsmodels.tools.sm_exceptions import ConvergenceWarning as _CW
    warnings.simplefilter("ignore", _CW)
if not HAS_SM:
    print("!" * 60)
    print("! statsmodels YOK -> tek-yonlu ANOVA. Pist/arac KONTROL EDILMIYOR.")
    print("! R degerleri asagi yonde YANLI. Sonuclar on-izleme sayilir.")
    print("! conda install -c conda-forge statsmodels")
    print("!" * 60 + "\n")
M_multi = M[M[ID_COL].isin(multi)].copy()
print(f"Repeatability veri: {len(M_multi)} satir, {M_multi[ID_COL].nunique()} surucu "
      f"(>=2 arac kullananlar)\n")

rows = []
for i, m in enumerate(ALL_METRICS, 1):
    r = repeatability_lmm(M_multi, m)
    lo, hi, nb = bootstrap_R(M_multi, m)
    # permutasyon p: surucu etiketlerini karistir
    pval = np.nan
    if np.isfinite(r['R']):
        rg = np.random.default_rng(RNG_SEED + i)
        obs, null = r['R'], []
        for _ in range(R_PERM):
            d2 = M_multi.copy()
            d2[ID_COL] = rg.permutation(d2[ID_COL].values)
            rr = repeatability_lmm(d2, m)['R']
            if np.isfinite(rr):
                null.append(rr)
        if len(null) >= max(20, R_PERM // 5):
            pval = (np.sum(np.array(null) >= obs) + 1) / (len(null) + 1)
    rows.append(dict(metrik=m, boyut=METRIC2DIM[m], R=r['R'],
                     ci_lo=lo, ci_hi=hi, ci_genislik=(hi - lo) if np.isfinite(hi) else np.nan,
                     var_surucu=r['var_g'], var_artik=r['var_e'],
                     yontem=r['method'], yakinsadi=r.get('converged'),
                     sinir=r.get('boundary', False),
                     n=r['n'], p_perm=pval, n_boot=nb))
    _eta = (time.time() - t0) / i * (len(ALL_METRICS) - i)
    _eta = (time.time() - t0) / i * (len(ALL_METRICS) - i)
    print(f"  [{i:>2}/19] {m:<24} R={r['R'] if np.isfinite(r['R']) else float('nan'):.3f} "
          f"GA=[{lo:.3f},{hi:.3f}] p={pval if np.isfinite(pval) else float('nan'):.3f} "
          f"({r['method']}) ETA~{_eta/60:.1f}dk")

rep = pd.DataFrame(rows)
rej, q = bh_fdr(rep['p_perm'].values)
rep['q_bh'] = q; rep['fdr_anlamli'] = rej
rep['yuksek'] = (rep['R'] > 0.50) & rep['fdr_anlamli']

print(f"\nSure: {time.time()-t0:.1f}s")
print("\n" + "=" * 68)
print("  REPEATABILITY SONUCLARI (R'ye gore sirali)")
print("=" * 68)
print(rep.sort_values('R', ascending=False)[
    ['metrik', 'boyut', 'R', 'ci_lo', 'ci_hi', 'p_perm', 'q_bh',
     'fdr_anlamli', 'yuksek', 'yakinsadi', 'sinir']
].to_string(index=False))

# --- YAKINSAMA RAPORU (sessiz hata onleme) ---
_nc = rep[(rep['yakinsadi'] == False)]
_bd = rep[rep['sinir'] == True]
print(f"\n  Yakinsama: {int((rep['yakinsadi'] == True).sum())}/19 tamam · "
      f"{len(_nc)} YAKINSAMADI · {len(_bd)} sinir vakasi (varyans ~0)")
if len(_nc):
    print("  YAKINSAMAYAN metrikler — R degerleri GUVENILMEZ, makaleye giremez:")
    print("    " + ", ".join(_nc['metrik'].tolist()))
if len(_bd):
    print("  Sinir vakasi (surucu varyansi fiilen sifir — R=0 gercek olabilir):")
    print("    " + ", ".join(_bd['metrik'].tolist()))
if _lmm_errors:
    print(f"  MixedLM istisnasi: {len(_lmm_errors)} -> {_lmm_errors[:3]}")

n_high = int(rep['yuksek'].sum())
print(f"\n  On-kayit esigini gecen (R>0.50 VE BH-FDR p<0.05): {n_high}/19")
REPEAT_HIGH = n_high > 0

print(f"\n  Kullanilan yontem: {rep['yontem'].value_counts().to_dict()}")
if (rep['yontem'] == 'anova_fallback').any():
    print("  UYARI: anova_fallback = pist/arac kontrol EDILMEDI -> R deflated.")
print("\n  Boyut bazinda medyan R:")
print(rep.groupby('boyut')['R'].agg(['median', 'min', 'max', 'count']).to_string())

## BLOK 7 — Duyarlilik

NB14'un tam olarak coktugu yer burasiydi (audi_r8 cikarilinca CV %107 -> %37).
Bu sefer test **onceden** yazildi:

1. **3-aracli alt kume** — yalnizca uc arac kullanan surucular
2. **Leave-one-car-out** — her arac sirayla cikarilir
3. **Leave-one-track-out** — her pist sirayla cikarilir
4. **Supheli kimlikler harici** — `A_12`/`A_21` onekli ve zaman-damgali kayitlar cikarilir

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 7 — Duyarlilik analizleri
# ══════════════════════════════════════════════════════════════
def headline(data, label, n_perm=3000):
    """Ana olcut: en genis arac ciftinde accuracy + p."""
    d = residualize_track(data, ALL_METRICS)
    best = None
    for ca, cb in combinations(CARS, 2):
        A, B, common = profiles_for_pair(d, ca, cb, ALL_METRICS)
        if A is None: continue
        if best is None or len(common) > best[0]:
            best = (len(common), ca, cb, A, B)
    if best is None:
        return dict(senaryo=label, n=0, car_a=None, car_b=None,
                    accuracy=np.nan, chance=np.nan, p_perm=np.nan, anlamli=False)
    n, ca, cb, A, B = best
    r = identifiability(A, B, n_perm=n_perm)
    return dict(senaryo=label, n=r['n'], car_a=ca, car_b=cb,
                accuracy=r['accuracy'], chance=r['chance'], idiff=r['idiff'],
                p_perm=r['p_perm'],
                anlamli=bool(np.isfinite(r['p_perm']) and r['p_perm'] < ALPHA))

sens = [headline(M, 'TAM VERI')]

# 1) 3-aracli alt kume
if len(tri) >= 3:
    sens.append(headline(M[M['driver_id'].isin(tri)], f'3-ARACLI (n={len(tri)})'))
else:
    print(f"3-aracli surucu sayisi {len(tri)} — alt kume analizi atlandi.")

# 2) Leave-one-car-out
for c in CARS:
    sens.append(headline(M[M['car'] != c], f'LOCO: -{c[:12]}'))

# 3) Leave-one-track-out
for t in TRACKS:
    sens.append(headline(M[M['track'] != t], f'LOTO: -{t[:12]}'))

# 4) Supheli kimlikler harici
def is_suspect(did):
    if any(p in did for p in SUSPECT_PATTERNS):
        return True
    if SUSPECT_TIMESTAMPED:
        parts = did.split('_')
        if len(parts) >= 3 and parts[0].isdigit() and parts[1].isdigit() and len(parts[1]) == 6:
            return True
    return False

suspects = [d for d in M['driver_id'].unique() if is_suspect(d)]
print(f"Supheli kimlikler ({len(suspects)}): {suspects}")
if suspects:
    sens.append(headline(M[~M['driver_id'].isin(suspects)], f'SUPHELI HARIC (-{len(suspects)})'))

sens_df = pd.DataFrame(sens)
print("\n" + "=" * 68)
print("  DUYARLILIK TABLOSU")
print("=" * 68)
print(sens_df.to_string(index=False))

# Coklu bakis duzeltmesi: 8 senaryoya BH-FDR
_rej, _q = bh_fdr(sens_df['p_perm'].values)
sens_df['q_bh'] = _q
sens_df['anlamli_bh'] = _rej
print("\n  COKLU BAKIS DUZELTMESI (BH-FDR, 8 senaryo):")
print(sens_df[['senaryo', 'n', 'accuracy', 'p_perm', 'q_bh', 'anlamli', 'anlamli_bh']]
      .to_string(index=False))
_n_bh = int(sens_df['anlamli_bh'].sum())
print(f"\n  Duzeltme sonrasi anlamli senaryo: {_n_bh}/{len(sens_df)}")
if _n_bh == 0 and sens_df['anlamli'].sum() > 0:
    print("  UYARI: Duzeltmesiz anlamli gorunen senaryolar coklu-bakis")
    print("         duzeltmesini GECMIYOR. Tek tek yorumlanamazlar.")

base_row = sens_df.iloc[0]
flips = sens_df[(sens_df['anlamli'] != base_row['anlamli']) &
                (sens_df.index != 0)]
SENS_ROBUST = len(flips) == 0
print(f"\n  Ana sonucun isareti degisen senaryo: {len(flips)}")
if len(flips):
    print(flips[['senaryo', 'n', 'accuracy', 'p_perm', 'anlamli']].to_string(index=False))
print(f"  DUYARLILIK: {'SAGLAM' if SENS_ROBUST else 'KIRILGAN — raporlanacak'}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 8 — Betimsel CV (NB14 koprusu)
# ══════════════════════════════════════════════════════════════
print("NB14 ile karsilastirilabilirlik icin: her surucunun araclari arasi CV,")
print("ham degerler uzerinden (NB14 ile ayni: ddof=0).\n")

cv_rows = []
for did, g in M[M['driver_id'].isin(multi)].groupby('driver_id'):
    per_car = g.groupby('car')[ALL_METRICS].mean()
    if len(per_car) < 2:
        continue
    for m in ALL_METRICS:
        cv_rows.append(dict(driver_id=did, metrik=m, boyut=METRIC2DIM[m],
                            n_car=len(per_car), cv=cv_pct(per_car[m].values)))
cvdf = pd.DataFrame(cv_rows)

if len(cvdf):
    print("Boyut bazinda CV dagilimi (tum surucular, metrikler birlestirilmis):")
    print(cvdf.groupby('boyut')['cv'].describe()[['count','50%','25%','75%']]
          .rename(columns={'50%':'medyan','25%':'q1','75%':'q3'}).round(1).to_string())
    print("\nMetrik bazinda medyan CV (en dusuk 8 = en arac-degismez):")
    med = cvdf.groupby(['metrik','boyut'])['cv'].median().reset_index().sort_values('cv')
    print(med.head(8).to_string(index=False))
    print("\nEn yuksek 5 (en arac-bagimli):")
    print(med.tail(5).to_string(index=False))
    print("\nNB14 (n=1, Efe) karsilastirmasi:")
    print("  NB14: B1=%6.1  B2=%107.2  B3=%104.0  B4=%22.6  (boyut ORTALAMASI uzerinden)")
    print("  NB16: yukaridaki tablo METRIK bazinda — dogrudan karsilastirilamaz,")
    print("        cunku NB14'un boyut skoru tek metrik tarafindan domine ediliyordu.")
else:
    print("CV hesaplanamadi (yeterli cok-aracli surucu yok).")

## BLOK 10 — Kimlik merdiveni: Basamak 2 (gun-arasi, ayni arac)


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 10 — KIMLIK MERDIVENI: Basamak 2 (gun-arasi, AYNI arac)
# ══════════════════════════════════════════════════════════════
# Bu basamak makalenin YORUM MENTESESI: parmak izi KISIYI mi yoksa
# KAYIT OTURUMUNU mu yakaliyor?  Nisan 2026'da tespit edilen catal:
#   "ya ayni kisi oturumlar arasi gercekten degisiyor (K1'i destekler),
#    ya da parmak izi kisiyi degil oturumu yakaliyor (K2'yi curutur)."
# ON-KAYIT: KESIF etiketi, bootstrap %95 GA, betimsel. Manset DEGIL.
print("=" * 68)
print("  BASAMAK 2 — gun-arasi, AYNI arac  [KESIF]")
print("=" * 68)

rung2_rows = []
for car in CARS:
    sub = M_loo[M_loo.car == car]
    if sub.empty:
        continue
    # her kisi icin oturum-gunlerini ikiye bol; iki profil uret
    A_rows, B_rows, persons = [], [], []
    for pid, g in sub.groupby(ID_COL):
        sess = sorted(g['driver_id'].unique())
        if len(sess) < 2:
            continue
        half = len(sess) // 2
        s1, s2 = sess[:max(1, half)], sess[max(1, half):]
        A_rows.append(g[g['driver_id'].isin(s1)][ALL_METRICS].mean().values)
        B_rows.append(g[g['driver_id'].isin(s2)][ALL_METRICS].mean().values)
        persons.append(pid)
    if len(persons) < 3:
        print(f"  {car:<16} n={len(persons)}  -> YETERSIZ (>=3 gerekli)")
        rung2_rows.append(dict(car=car, n=len(persons), accuracy=np.nan,
                               chance=np.nan, p_perm=np.nan, idiff=np.nan))
        continue
    both = np.vstack([np.array(A_rows), np.array(B_rows)])
    Z = zscore_cols(both)
    n = len(persons)
    r = identifiability(Z[:n], Z[n:], n_perm=N_PERM, method='corr')
    print(f"  {car:<16} n={n:>2}  acc={r['accuracy']:.3f}  sans={r['chance']:.3f}  "
          f"Idiff={r['idiff']:+.1f}  p={r['p_perm']:.4f}   kisiler={persons}")
    rung2_rows.append(dict(car=car, n=n, accuracy=r['accuracy'], chance=r['chance'],
                           idiff=r['idiff'], p_perm=r['p_perm'], persons=persons))

rung2 = pd.DataFrame(rung2_rows)
_valid2 = rung2[rung2['n'] >= 3]
RUNG2_N = int(_valid2['n'].max()) if len(_valid2) else 0
RUNG2_STRONG = bool(len(_valid2) and (_valid2['p_perm'] < ALPHA).any())
print(f"\nBasamak 2 hukmu: {'GUCLU' if RUNG2_STRONG else 'ZAYIF'}  "
      f"(en buyuk n={RUNG2_N})")
print("UYARI: kucuk n -> KESIF; tavan kestirimi genis araliklidir, manset yapilamaz.")


## BLOK 11 — S1 (boyutsuz vs boyutlu) + S2 (turetilmis oranlar)


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 11 — S1: boyutsuz vs boyutlu  |  S2: turetilmis oran metrikleri
# ══════════════════════════════════════════════════════════════
# Her iki blok da AYRI FDR butcesiyle kosar ve BIRINCIL HUKMU DEGISTIREMEZ.
_dimless = [m for m in ALL_METRICS if m in DIMENSIONLESS]
_dimful  = [m for m in ALL_METRICS if m not in DIMENSIONLESS]
print("=" * 68)
print("  S1 — a priori metrik sinifi (veriye BAKILMADAN sabitlendi)")
print("=" * 68)
print(f"  BOYUTSUZ ({len(_dimless)}): {_dimless}")
print(f"  BOYUTLU  ({len(_dimful)}): {_dimful}")

_R = {}
for m in ALL_METRICS:
    try:
        _R[m] = repeatability_lmm(M, m)['R']
    except Exception:
        _R[m] = np.nan
_a = [ _R[m] for m in _dimless if np.isfinite(_R.get(m, np.nan)) ]
_b = [ _R[m] for m in _dimful  if np.isfinite(_R.get(m, np.nan)) ]
from scipy.stats import mannwhitneyu
if len(_a) >= 3 and len(_b) >= 3:
    _u, _p = mannwhitneyu(_a, _b, alternative='greater')
    S1_RESULT = dict(n_dimensionless=len(_a), n_dimensional=len(_b),
                     median_R_dimensionless=float(np.median(_a)),
                     median_R_dimensional=float(np.median(_b)),
                     U=float(_u), p=float(_p), significant=bool(_p < ALPHA))
else:
    S1_RESULT = dict(n_dimensionless=len(_a), n_dimensional=len(_b), p=np.nan,
                     significant=False, note='yetersiz metrik')
# v5: DEJENERE DURUM UYARISI — R'lerin cogu tam 0 ise test BILGI TASIMAZ.
# "anlamsiz" demek gercek bir null ima eder; oysa burada tahmin edici sinir
# vakasina takilmis olabilir. Ikisini karistirmamak sart.
_all = [x for x in list(_a) + list(_b) if np.isfinite(x)]
_zero_share = float(np.mean([x == 0.0 for x in _all])) if _all else np.nan
S1_RESULT['zero_R_share'] = _zero_share
S1_RESULT['uninformative'] = bool(np.isfinite(_zero_share) and _zero_share >= 0.75)
if S1_RESULT['uninformative']:
    S1_RESULT['significant'] = False
    print(f"\n  !!! S1 BILGI TASIMIYOR: metriklerin %{_zero_share*100:.0f}'i tam R=0 donduruyor.")
    print("      Bu bir 'fark yok' bulgusu DEGIL; tahmin edici sinir vakasinda.")
    print("      Levene ciktisina ve MixedLM yakinsama raporuna bak.")
print(f"\n  medyan R  boyutsuz={S1_RESULT.get('median_R_dimensionless', np.nan):.3f}  "
      f"boyutlu={S1_RESULT.get('median_R_dimensional', np.nan):.3f}  "
      f"p={S1_RESULT.get('p', np.nan):.4f}  "
      f"{'ANLAMLI' if S1_RESULT.get('significant') else 'anlamsiz'}")

print("\n" + "=" * 68)
print("  S2 — turetilmis oransal metrikler")
print("=" * 68)
# GECERLILIK KAPISI (on-kayitli): hucre-ici varyansi ~0 olan metrik pist
# geometrisi sabitidir, kimlik tasimaz -> DUSER.
S2_ROWS, S2_DROPPED = [], []
for m in DERIVED_METRICS:
    if m not in M.columns or M[m].notna().sum() < 6:
        S2_DROPPED.append((m, 'uretilmedi/yetersiz')); continue
    _wc = M.groupby(['car', 'track'])[m].std(ddof=0)
    if float(np.nanmax(_wc.values)) < 1e-9:
        S2_DROPPED.append((m, 'hucre-ici varyans ~0 -> pist geometrisi sabiti')); continue
    try:
        _r = repeatability_lmm(M, m)
        _lo, _hi, _nb = bootstrap_R(M, m, n_boot=N_BOOT)   # tuple doner: (lo, hi, n)
    except Exception as e:
        S2_DROPPED.append((m, f'hesaplanamadi: {type(e).__name__}')); continue
    S2_ROWS.append(dict(metric=m, R=_r['R'], ci_lo=_lo, ci_hi=_hi, n_boot_ok=_nb,
                        method=_r.get('method', '?'),
                        gecti=bool(np.isfinite(_lo) and _lo > 0.40)))
S2 = pd.DataFrame(S2_ROWS)
if len(S2):
    print(S2.to_string(index=False))
    print(f"\n  On-kayit esigini (GA alt siniri > 0.40) gecen: "
          f"{int(S2['gecti'].sum())}/{len(S2)}")
else:
    print("  Hicbir turetilmis metrik degerlendirilemedi.")
for m, why in S2_DROPPED:
    print(f"  DUSTU  {m:<20} {why}")


## BLOK 12 — On-kayit hukmu (uc dal)


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 12 — ON-KAYIT HUKMU (uc dal, KOSUDAN ONCE yazildi)
# ══════════════════════════════════════════════════════════════
_valid = ident_B[ident_B['n'] >= 3] if 'ident_B' in dir() else pd.DataFrame()
# v5.2 DUZELTME: on-kayit "arac ciftleri arasinda BH-FDR" diyor; v5.1 ham p
# kullaniyordu ve bu BIRINCIL hukmu ters ceviriyordu (EXC: ham p ile anlamli,
# FDR ile degil). Coklu bakis duzeltmesi olmadan hukum verilmez.
if len(_valid):
    # v5.3 DUZELTME: bh_fdr (reddedildi_mi, duzeltilmis_p) DONDURUR.
    # v5.2'de _q[0] alinip maske q sanilmisti -> boolean 0/1'e donusunce mantik
    # TERSINE dondu (p=0.19 "FDR gecer", p=0.0075 "gecmez"). Diger iki kullanim
    # yerinde 'rej, q = bh_fdr(...)' bicimi zaten dogruydu.
    _rej3, _q3 = bh_fdr(_valid['p_perm'].values, alpha=ALPHA)
    _valid = _valid.assign(q_bh=_q3, fdr_rej=_rej3)
    RUNG3_SIG = bool(_rej3.any())
    print("  Basamak 3 — BH-FDR (arac ciftleri arasi):")
    for _, _r in _valid.iterrows():
        print(f"    {_r['car_a'][:12]:<12} x {_r['car_b'][:12]:<12} n={int(_r['n']):>2} "
              f"p={_r['p_perm']:.4f}  q={_r['q_bh']:.4f}  "
              f"{'FDR GECER' if _r['fdr_rej'] else 'FDR gecmez'}")
else:
    RUNG3_SIG = False
_best = (_valid.sort_values('n', ascending=False).iloc[0]
         if len(_valid) else None)

# ── v5.1: BASAMAK 2 GUC KAPISI ──
# n=4'te sans 0.25; p<0.05 icin dordun dordu dogru olmali. Boyle bir testte
# "ZAYIF" cikmasi KANIT DEGIL, olculemezliktir. v5'te bu kapi yoktu ve hukum
# gucsuz bir basamaga takilip V4_TUTARSIZ verdi. On-kayit Basamak 2'yi zaten
# "KESIF" ilan etmisti; kod onu gate yapmakla on-kayitla CELISIYORDU.
_min_n_for_power = int(np.ceil(1 / ALPHA))
RUNG2_POWERED = bool(RUNG2_N >= _min_n_for_power)
if not RUNG2_POWERED:
    print(f"  UYARI: Basamak 2 en buyuk n={RUNG2_N} < {_min_n_for_power} -> "
          f"hicbir sonuc p<{ALPHA} olamaz. OLCULEMEDI olarak raporlanir.")

if not RUNG2_POWERED:
    if RUNG3_SIG:
        VERDICT_CODE, VERDICT_TEXT = 'V2_KOSULLU', (
            'Araclar arasi kimlik ANLAMLI. Basamak 2 gucsuz oldugu icin '
            '"kisi mi oturum mu" sorusu ACIK; tavan olculemedi.')
    else:
        VERDICT_CODE, VERDICT_TEXT = 'V3_KOSULLU', (
            'Araclar arasi kimlik anlamsiz VE Basamak 2 gucsuz. Bu tasarimda '
            'transfer gosterilemedi; null yoruma kapali, guc-sinirli.')
elif RUNG2_STRONG and RUNG3_SIG:
    VERDICT_CODE, VERDICT_TEXT = 'V1', ('K2 dogrulandi — stil cekirdegi arac-degismez.')
elif RUNG2_STRONG and not RUNG3_SIG:
    VERDICT_CODE, VERDICT_TEXT = 'V2', (
        'Parmak izi KISIYI yakaliyor, ARAC TRANSFERI kiriyor. '
        'Arac-basina referans zorunlulugunun ampirik gerekcesi.')
elif (not RUNG2_STRONG) and not RUNG3_SIG:
    VERDICT_CODE, VERDICT_TEXT = 'V3', (
        'Parmak izi OTURUM duzeyinde; araclar-arasi null YORUMA KAPALI. '
        'Bulgu, olcum-gecerliligi bulgusuna donusur.')
else:
    VERDICT_CODE, VERDICT_TEXT = 'V4_TUTARSIZ', (
        'Ic tutarsizlik (Basamak 2 zayif ama Basamak 3 anlamli). '
        'YORUMLANMAZ — olcum hatasi olarak sorusturulacak.')

print("=" * 68)
print(f"  ON-KAYIT HUKMU: {VERDICT_CODE}")
print("=" * 68)
print(f"  Basamak 2 (gun-arasi) : "
      f"{'GUCLU' if RUNG2_STRONG else ('ZAYIF' if RUNG2_POWERED else 'OLCULEMEDI (guc yok)')}"
      f"   [en buyuk n={RUNG2_N}]")
print(f"  Basamak 3 (arac-arasi): {'ANLAMLI' if RUNG3_SIG else 'ANLAMSIZ'}")
if _best is not None:
    print(f"  En buyuk cift: {_best['car_a']} x {_best['car_b']}  n={int(_best['n'])}  "
          f"acc={_best['accuracy']:.3f}  sans={_best['chance']:.3f}  p={_best['p_perm']:.4f}")
print(f"\n  {VERDICT_TEXT}")
print("\n  Durdurma kurali: hukum hangi dala duserse o yazilir. Alt kume aramasi yok,")
print("  esik degistirme yok, 'su senaryoda anlamli cikti' cumlesi yok.")

# Levene — pist-ici artik varyans homojenligi (v5 DUZELTME 3)
print("\n" + "-" * 68)
print("  LEVENE — pist-ici artik varyans homojenligi")
print("-" * 68)
LEVENE_INFO = {}
for m in ALL_METRICS[:6]:
    lv = levene_by_track(M, m)
    LEVENE_INFO[m] = lv
    print(f"  {m:<24} W={lv['stat']:>7.3f}  p={lv['p']:.4f}  "
          f"{'HETEROSKEDASTIK' if np.isfinite(lv['p']) and lv['p'] < ALPHA else 'homojen'}")


## BLOK 9 — Gorseller

Proje karari: Okabe-Ito paleti (CVD-guvenli, Wong 2011), scienceplots, PDF+PNG cift cikti,
**radar chart yok** (Duan 2023 polygon-area bias nedeniyle destekleyiciye indirilmisti).
Eksen etiketleri Ingilizce — dogrudan makaleye girecekler.

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 9 — Gorseller (5 figur)
# ══════════════════════════════════════════════════════════════

# --- Figur 1: Identifiability matrisi ---
try:
    if BEST is not None:
        A, B, common = profiles_for_pair(M_res, CA, CB, ALL_METRICS)
        S = similarity_matrix(A, B)
        fig, ax = plt.subplots(figsize=(6.2, 5.2))
        im = ax.imshow(S, cmap=CMAP_DIV, vmin=-1, vmax=1, aspect='equal')
        for i in range(len(common)):
            ax.add_patch(plt.Rectangle((i-.5, i-.5), 1, 1, fill=False,
                                       edgecolor='black', lw=1.1))
        short = [d.split('_')[-1][:6] for d in common]
        ax.set_xticks(range(len(common))); ax.set_xticklabels(short, rotation=90, fontsize=6)
        ax.set_yticks(range(len(common))); ax.set_yticklabels(short, fontsize=6)
        ax.set_xlabel(f'Target: {CB}'); ax.set_ylabel(f'Database: {CA}')
        ax.set_title('(a) Cross-car identifiability matrix', fontsize=10, fontweight='bold')
        fig.colorbar(im, ax=ax, shrink=0.8, label='Profile similarity (r)')
        save_fig(fig, 'fig1_identifiability_matrix'); plt.show()
except Exception as e:
    print(f"  ! Figur 1 atlandi: {e}")

# --- Figur 2: Permutasyon null + gozlenen ---
try:
    key = f"B_pist_artik|{CA}|{CB}"
    null = nulls_B.get(key)
    if null is not None:
        obs = float(valid_B[(valid_B.car_a == CA) & (valid_B.car_b == CB)]['accuracy'].iloc[0])
        pv  = float(valid_B[(valid_B.car_a == CA) & (valid_B.car_b == CB)]['p_perm'].iloc[0])
        fig, ax = plt.subplots(figsize=(6.2, 3.4))
        ax.hist(null, bins=30, color='#999999', edgecolor='white', label='Permutation null')
        ax.axvline(obs, color='#D55E00', lw=2.2, label=f'Observed = {obs:.3f}')
        ax.axvline(1/len(common), color='#0072B2', ls='--', lw=1.4,
                   label=f'Chance = {1/len(common):.3f}')
        ax.set_xlabel('Identification accuracy'); ax.set_ylabel('Count')
        ax.set_title(f'(b) Permutation test  ($p$ = {pv:.4f}, {N_PERM:,} permutations)',
                     fontsize=10, fontweight='bold')
        ax.legend(fontsize=7)
        save_fig(fig, 'fig2_permutation_null'); plt.show()
except Exception as e:
    print(f"  ! Figur 2 atlandi: {e}")

# --- Figur 3: Forest plot (repeatability) ---
try:
    r2 = rep.dropna(subset=['R']).sort_values(['boyut', 'R'])
    fig, ax = plt.subplots(figsize=(6.6, 6.4))
    for i, (_, row) in enumerate(r2.iterrows()):
        col = DIM_COLORS.get(row['boyut'], '#333333')
        if np.isfinite(row['ci_lo']):
            ax.plot([row['ci_lo'], row['ci_hi']], [i, i], color=col, lw=1.6, alpha=.65)
        ax.plot(row['R'], i, 'o', color=col, ms=6,
                mec='black' if row['fdr_anlamli'] else 'none', mew=.9)
    ax.axvline(0.5, color='gray', ls=':', lw=1)
    ax.set_yticks(range(len(r2)))
    ax.set_yticklabels([f"{r.metrik}" for r in r2.itertuples()], fontsize=7)
    ax.set_xlabel('Adjusted repeatability $R$ (95% bootstrap CI)')
    ax.set_xlim(-0.05, 1.05)
    _m3 = rep['yontem'].mode().iloc[0] if len(rep) else 'unknown'
    ax.set_title('(c) Per-metric repeatability across vehicles' +
                 ('' if _m3 == 'mixedlm' else '  [UNADJUSTED]'),
                 fontsize=10, fontweight='bold',
                 color=('black' if _m3 == 'mixedlm' else '#D55E00'))
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([0],[0],marker='o',ls='',color=c,label=d)
                       for d, c in DIM_COLORS.items()],
              fontsize=7, loc='lower right', title='Dimension', title_fontsize=7)
    save_fig(fig, 'fig3_repeatability_forest'); plt.show()
except Exception as e:
    print(f"  ! Figur 3 atlandi: {e}")

# --- Figur 4: Yigilmis varyans paylari ---
try:
    v = rep.dropna(subset=['var_surucu', 'var_artik']).copy()
    v['tot'] = v['var_surucu'] + v['var_artik']
    v = v[v['tot'] > 0].sort_values(['boyut', 'R'])
    fig, ax = plt.subplots(figsize=(6.6, 4.2))
    x = np.arange(len(v))
    ds = (v['var_surucu'] / v['tot']).values
    ax.bar(x, ds, color='#0072B2', label='Driver', edgecolor='white', lw=.4)
    ax.bar(x, 1 - ds, bottom=ds, color='#E69F00', label='Residual', edgecolor='white', lw=.4)
    ax.set_xticks(x); ax.set_xticklabels(v['metrik'], rotation=90, fontsize=6)
    ax.set_ylabel('Variance share'); ax.set_ylim(0, 1)
    _meth = rep['yontem'].mode().iloc[0] if len(rep) else 'unknown'
    _ttl = ('(d) Variance decomposition (track and car as fixed effects)'
            if _meth == 'mixedlm'
            else '(d) Variance decomposition (UNADJUSTED one-way; track/car NOT controlled)')
    ax.set_title(_ttl, fontsize=10, fontweight='bold',
                 color=('black' if _meth == 'mixedlm' else '#D55E00'))
    ax.legend(fontsize=7, ncol=2)
    save_fig(fig, 'fig4_variance_shares'); plt.show()
except Exception as e:
    print(f"  ! Figur 4 atlandi: {e}")

# --- Figur 5: Duyarlilik slope ---
try:
    s = sens_df.dropna(subset=['accuracy']).copy()
    fig, ax = plt.subplots(figsize=(6.6, 3.8))
    xs = np.arange(len(s))
    cols = ['#009E73' if a else '#D55E00' for a in s['anlamli']]
    ax.bar(xs, s['accuracy'], color=cols, edgecolor='white', lw=.5)
    ax.plot(xs, s['chance'], 'k--', lw=1.2, label='Chance level')
    for i, (acc, p) in enumerate(zip(s['accuracy'], s['p_perm'])):
        if np.isfinite(p):
            ax.text(i, acc + .012, f"p={p:.3f}", ha='center', fontsize=6, rotation=90)
    ax.set_xticks(xs); ax.set_xticklabels(s['senaryo'], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Identification accuracy')
    ax.set_title('(e) Sensitivity analyses', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)
    save_fig(fig, 'fig5_sensitivity'); plt.show()
except Exception as e:
    print(f"  ! Figur 5 atlandi: {e}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOK 10 — Senaryo hukmu + nb16_info.json
# ══════════════════════════════════════════════════════════════
from datetime import datetime

if IDENT_SIGNIFICANT and REPEAT_HIGH:
    SCEN, TXT = "S1", ("K2 dogrulandi + mekanizma belirlendi. "
                       "Parmak izi araclar arasi surucuyu tanimliyor VE hangi "
                       "metriklerin bunu sagladigi belirlenebiliyor.")
elif IDENT_SIGNIFICANT and not REPEAT_HIGH:
    SCEN, TXT = "S2", ("Kombinasyon tanimliyor, tek metrik degil. Cok-degiskenli imza "
                       "bulgusu; tek-metrik CV yaklasiminin neden yaniltici oldugunu gosterir.")
elif (not IDENT_SIGNIFICANT) and REPEAT_HIGH:
    SCEN, TXT = "S3", ("Metrikler surucuye ozgu ama bilesik parmak izi tanimlamiyor -> "
                       "4-boyut toplulastirmasi sinyali yok ediyor olabilir. Bulgu K2 "
                       "hakkinda degil, fingerprint mimarisi hakkinda.")
else:
    SCEN, TXT = "S4", ("K2 duser. ACGym populasyonunda araclar arasi degismezlik "
                       "gosterilemedi; negatif bulgu olarak Discussion'da kalir.")

print("=" * 68)
print(f"  ON-KAYIT HUKMU: {SCEN}")
print("=" * 68)
print(f"  Identifiability anlamli : {IDENT_SIGNIFICANT}")
print(f"  Repeatability yuksek    : {REPEAT_HIGH}  ({n_high}/19 metrik)")
print(f"  Duyarlilik saglam       : {SENS_ROBUST}")
print(f"\n  {TXT}")
if not SENS_ROBUST:
    print("\n  UYARI: Duyarlilik testlerinde isaret degisimi var. Ana sonuc kirilgan;")
    print("         makalede bu acikca raporlanacak.")

info = dict(
    identity_policy=POLICY,
    calibration=CALIB,
    rung3_bh_table=(_valid[['car_a','car_b','n','p_perm','q_bh','fdr_rej']].to_dict('records')
                    if 'q_bh' in getattr(_valid, 'columns', []) else None),
    calibration_ok=bool(CALIB_OK),
    rung2_powered=bool(RUNG2_POWERED),
    rung2_max_n=int(RUNG2_N),
    lmm_optimizer='powell (v5.1; lbfgs tekil matris veriyordu)',
    amendment='v5.1: naif WCCN sizdiriyordu (null FPR %88-97) -> LOO-WCCN; MixedLM lbfgs->powell; Basamak 2 guc kapisi',
    identifiability_v4_zscore=(ident_B_v4.to_dict('records') if 'ident_B_v4' in dir() else None),
    descriptive_cv=(cvdf.to_dict('records') if 'cvdf' in dir() else None),
    ablock_excluded=(POLICY == 'P_EXC'),
    exclude_ids=EXCLUDE_IDS,
    id_col=ID_COL,
    ladder=LADDER_INFO,
    rung2=(rung2.to_dict('records') if 'rung2' in dir() else None),
    rung2_strong=bool(RUNG2_STRONG),
    rung3_significant=bool(RUNG3_SIG),
    verdict_code=VERDICT_CODE,
    verdict_text=VERDICT_TEXT,
    s1_dimensionless_vs_dimensional=S1_RESULT,
    s2_derived=(S2.to_dict('records') if len(S2) else []),
    s2_dropped=[{'metric': m, 'reason': w} for m, w in S2_DROPPED],
    levene=LEVENE_INFO,
    r_perm=R_PERM,
    tdd_orta_median_R=(None if not np.isfinite(_mid_med) else float(_mid_med)),
    notebook="NB16_cross_car_identifiability_v5",
    run_timestamp=datetime.now().isoformat(timespec='seconds'),
    tier_source=str(TIER_DIR),
    degenerate_ids_excluded=DEGENERATE_IDS,
    suspect_ids=suspects,
    n_perm=N_PERM, n_boot=N_BOOT, alpha=ALPHA, seed=RNG_SEED,
    tdd_pass=bool(TDD_PASS),
    design=DESIGN_INFO,
    apriori_precision_bonett=_prec,
    identifiability_pooled=ident_B.to_dict('records'),
    identifiability_within_track=(ident_A.to_dict('records') if len(ident_A) else []),
    identifiability_summary=dict(pooled=sB, within_track=sA),
    dimension_identifiability=(dim_ident.to_dict('records') if len(dim_ident) else []),
    leave_one_metric_out=(lomo.to_dict('records') if len(lomo) else []),
    repeatability=rep.to_dict('records'),
    n_metrics_high_R=int(n_high),
    n_converged=int((rep['yakinsadi'] == True).sum()),
    n_not_converged=int((rep['yakinsadi'] == False).sum()),
    n_boundary=int((rep['sinir'] == True).sum()),
    non_converged_metrics=rep.loc[rep['yakinsadi'] == False, 'metrik'].tolist(),
    sensitivity=sens_df.to_dict('records'),
    sensitivity_robust=bool(SENS_ROBUST),
    scenario=SCEN, scenario_text=TXT,
    statsmodels_available=bool(HAS_SM),
    repeatability_adjusted=bool(HAS_SM and (rep['yontem'] == 'mixedlm').any()),
    sim_method_note=("k>=8 korelasyon, k<8 negatif Oklid; boyut duzeyi her zaman negeuc"),
    sensitivity_bh_significant=int(sens_df['anlamli_bh'].sum()),
)

def _clean(o):
    if isinstance(o, dict):  return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, list):  return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return (None if not np.isfinite(o) else float(o))
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, float) and not np.isfinite(o): return None
    return o

with open(OUT_DIR / "nb16_info.json", "w", encoding="utf-8") as f:
    json.dump(_clean(info), f, indent=2, ensure_ascii=False)

rep.to_csv(OUT_DIR / "repeatability_per_metric.csv", index=False)
ident_B.to_csv(OUT_DIR / "identifiability_pooled.csv", index=False)
sens_df.to_csv(OUT_DIR / "sensitivity.csv", index=False)
if len(cvdf):
    cvdf.to_csv(OUT_DIR / "descriptive_cv.csv", index=False)

print("\n" + "=" * 68)
print("  KAYDEDILEN DOSYALAR")
print("=" * 68)
for f in sorted(OUT_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name:<38} {f.stat().st_size/1024:>8.1f} KB")
print(f"  figures/  -> {len(list(FIG_DIR.glob('*.pdf')))} PDF + "
      f"{len(list(FIG_DIR.glob('*.png')))} PNG")
print("\nNB16 tamamlandi. nb16_info.json'u paylas — paper_numbers oradan beslenecek.")

In [ ]:
import json, glob, os

paths = sorted(glob.glob(str(TRACK_ROOT / "data" / "fingerprints" / "t*_*" / "t*_info.json"))) \
      + sorted(glob.glob(str(TRACK_ROOT / "results" / "cross_car_*" / "nb16_info.json")))

for p in paths:
    d = json.load(open(p, encoding='utf-8'))
    print('=' * 70)
    print(os.path.relpath(p, TRACK_ROOT))
    for k, v in d.items():
        s = repr(v)
        print('   {:<28} {}'.format(k, s if len(s) <= 120 else s[:120] + ' ...'))

In [ ]:
import json, glob, os

pats = [str(TRACK_ROOT / "data" / "fingerprints" / "corner_type_profiles_*" / "nb7a7*.json"),
        str(TRACK_ROOT / "results" / "gap_analysis_*" / "nb15_info.json")]

for pat in pats:
    for p in sorted(glob.glob(pat)):
        d = json.load(open(p, encoding='utf-8'))
        print('=' * 72)
        print(os.path.relpath(p, TRACK_ROOT))
        print(json.dumps(d, indent=1, ensure_ascii=False))